In [ ]:
import os
import sys
import argparse
import pandas as pd
import torch
import anndata as ad
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from DeepRUOT.losses import OT_loss1
from DeepRUOT.utils import (
    generate_steps, load_and_merge_config,
    SchrodingerBridgeConditionalFlowMatcher,
    generate_state_trajectory, get_batch, get_batch_size
)
from DeepRUOT.train import train_un1_reduce, train_all
from DeepRUOT.models import FNet_interaction, scoreNet2
from DeepRUOT.constants import DATA_DIR, RES_DIR
from DeepRUOT.exp import setup_exp

## Load config and data

In [ ]:
config_path = '../config/mosta_config.yaml'

# Load and merge configuration
config = load_and_merge_config(config_path)

df = pd.read_csv(os.path.join(DATA_DIR, config['data']['file_path']))
df = df.iloc[:, :config['data']['dim'] + 1]
#df = df[df.iloc[:,1] > 0.4]
device = torch.device('cpu')
exp_dir, logger = setup_exp(
            RES_DIR, 
            config, 
            config['exp']['name']
        )
dim = config['data']['dim']

model_config = config['model']
        
f_net = FNet_interaction(
            in_out_dim=model_config['in_out_dim'],
            hidden_dim=model_config['hidden_dim'],
            n_hiddens=model_config['n_hiddens'],
            activation=model_config['activation'],
            use_spatial = True, 
            num_heads = 8,
            thre = 0.06,
            num_layers = 1,

        ).to(device)

sf2m_score_model = scoreNet2(
    in_out_dim=model_config['in_out_dim'],
    hidden_dim=model_config['score_hidden_dim'],
    activation=model_config['activation']
).float().to(device)

f_net.load_state_dict(torch.load(os.path.join(exp_dir, 'model_final'),map_location=torch.device('cpu')))
f_net.to(device)
sf2m_score_model.load_state_dict(torch.load(os.path.join(exp_dir, 'score_model'),map_location=torch.device('cpu')))
sf2m_score_model.to(device)

## Load Mouse Data

In [ ]:
import scanpy as sc

# 加载 h5ad 文件
adata = sc.read("../spatial_data/Mouse_embryo_all_stage.h5ad")

# 查看数据的基本信息
print(adata)

import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import distance

# 假设 adata 是您的原始 AnnData 对象
# 获取所有唯一的批次名称
batch_names = adata.obs['timepoint'].cat.categories

# 创建字典存储每个批次的 AnnData 对象，确保是实际对象
adata_dict = {}
for batch in batch_names:
    adata_dict[batch] = adata[adata.obs['timepoint'] == batch].copy()

# 定义预处理函数
def preprocess_adata(adata):
    #sc.pp.normalize_total(adata, target_sum=1e4)
    #sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    adata = adata[:, adata.var.highly_variable]
    return adata

# 定义空间坐标缩放函数
def scale_spatial_coords(adata):
    spatial_coords = adata.obsm['spatial']
    x_min, x_max = spatial_coords[:, 0].min(), spatial_coords[:, 0].max()
    y_min, y_max = spatial_coords[:, 1].min(), spatial_coords[:, 1].max()
    x_range = x_max - x_min
    spatial_coords[:, 0] = (spatial_coords[:, 0] - x_min) / x_range
    scale_factor = 1 / x_range
    spatial_coords[:, 1] = (spatial_coords[:, 1] - y_min) * scale_factor
    adata.obsm['spatial'] = spatial_coords # 更新修改后的坐标
    return adata

# 定义绘图函数
def plot_spatial(adata, batch_name):
    spatial_coords = adata.obsm['spatial']
    annotations = adata.obs['Annotation']
    annotation_colors = adata.uns['Annotation_colors']
    category_to_color = dict(zip(annotations.cat.categories, annotation_colors))
    colors = annotations.map(category_to_color)
    plt.figure(figsize=(10, 8))
    plt.scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=colors, s=10, alpha=1)
    plt.title(f'Spatial Visualization for {batch_name}')
    plt.xlabel('Scaled X')
    plt.ylabel('Scaled Y')
    plt.show()


def remove_outliers(adata, radius=0.05, threshold=5):
    # 获取空间坐标
    spatial_coords = adata.obsm['spatial']
    # 计算所有数据点之间的距离矩阵
    dist_matrix = distance.cdist(spatial_coords, spatial_coords)
    # 计算每个点的邻居数量（减去自身）
    neighbors = np.sum(dist_matrix < radius, axis=1) - 1
    # 创建掩码：保留邻居数量大于等于阈值的点
    mask = neighbors >= threshold
    # 应用掩码，删除离群点
    adata = adata[mask]
    return adata

# 自动化处理每个批次
for batch in batch_names:
    adata_batch = adata_dict[batch].copy()
    # adata_batch = preprocess_adata(adata_batch)
    # adata_batch = scale_spatial_coords(adata_batch.copy())
    # adata_batch = remove_outliers(adata_batch, radius=0.05, threshold=5)
    # adata_batch = scale_spatial_coords(adata_batch.copy())
    adata_dict[batch]=adata_batch.copy()

import numpy as np
import pandas as pd

# 初始化标签列表
labels_list = []
T = 5
batch_indices = [3, 4, 5, 6]

for t, batch_idx in enumerate(batch_indices):
    adata_t = adata_dict[batch_names[batch_idx]]
    labels_t = adata_t.obs['annotation'].values  # 获取当前时间点的 Annotation
    labels_list.append(labels_t)

# 合并所有标签
all_labels = np.concatenate(labels_list)

# 读取 CSV 文件
df_new = pd.read_csv('../data/mosta_four_time.csv')

# 添加 Annotation 列
df_new['Annotation'] = all_labels
# 获取 Annotation 的类别和颜色
if pd.api.types.is_categorical_dtype(adata.obs['annotation']):
    categories = adata.obs['annotation'].cat.categories
else:
    categories = adata.obs['annotation'].unique()  # 如果不是 categorical 类型

colors = adata.uns['annotation_colors']

# 创建标签到颜色的映射
label_to_color = dict(zip(categories, colors))

## Interpolation

In [ ]:
class TimeMapper:
    """真实时间（如E12.5）与模型时间（0-3）之间的映射，支持外推"""
    def __init__(self, real_times, model_times=None):
        """
        参数:
            real_times: 真实时间点列表，如 [12.5, 13.5, 14.5, 15.5]
            model_times: 对应的模型时间点，如 [0, 1, 2, 3]
        """
        self.real_times = np.array(sorted(real_times))
        if model_times is None:
            self.model_times = np.linspace(0, len(real_times)-1, len(real_times))
        else:
            self.model_times = np.array(model_times)
        
        assert len(self.real_times) == len(self.model_times), \
            "真实时间和模型时间数量必须相同"
        
        # 计算斜率用于线性外推
        self.slope = (self.model_times[-1] - self.model_times[0]) / \
                     (self.real_times[-1] - self.real_times[0])
    
    def real_to_model(self, real_time):
        """将真实时间转换为模型时间（线性插值+外推）"""
        real_time = np.atleast_1d(real_time)
        result = np.zeros_like(real_time, dtype=float)
        
        for i, t in enumerate(real_time):
            if t < self.real_times[0]:
                # 左外推
                result[i] = self.model_times[0] + self.slope * (t - self.real_times[0])
            elif t > self.real_times[-1]:
                # 右外推
                result[i] = self.model_times[-1] + self.slope * (t - self.real_times[-1])
            else:
                # 插值
                result[i] = np.interp(t, self.real_times, self.model_times)
        
        return result[0] if len(result) == 1 else result

    def model_to_real(self, model_time):
        """将模型时间转换为真实时间（线性插值+外推）"""
        model_time = np.atleast_1d(model_time)
        result = np.zeros_like(model_time, dtype=float)
        
        for i, t in enumerate(model_time):
            if t < self.model_times[0]:
                result[i] = self.real_times[0] + (t - self.model_times[0]) / self.slope
            elif t > self.model_times[-1]:
                result[i] = self.real_times[-1] + (t - self.model_times[-1]) / self.slope
            else:
                result[i] = np.interp(t, self.model_times, self.real_times)
        
        return result[0] if len(result) == 1 else result
    
    def get_model_times(self, real_time_list):
        """批量转换真实时间到模型时间"""
        return [self.real_to_model(t) for t in real_time_list]
    
    def get_real_times(self, model_time_list):
        """批量转换模型时间到真实时间"""
        return [self.model_to_real(t) for t in model_time_list]
    
    def print_mapping(self):
        """打印时间映射关系"""
        print("时间映射关系:")
        for real_t, model_t in zip(self.real_times, self.model_times):
            print(f"  E{real_t:.1f} <-> 模型时间 {model_t:.2f}")

class InterpolationConfig:
    """插值配置参数"""
    def __init__(self):
        # 时间点设置 - 使用真实时间（如E12.5对应12.5）
        self.real_times_to_interpolate = None  # 要插值的真实时间点，如 [11.5, 12.5, 14.0, 16.5]
        
        # 或者使用模型时间（0-3）
        self.start_time = 0.0      # 起始时间（模型时间）
        self.end_time = 2.0        # 结束时间（模型时间）
        self.num_frames = 14      # 总帧数（用于密集插值）
        self.target_times = None   # 直接指定模型时间点，如 [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
        
        # 时间映射器（用于真实时间和模型时间转换）
        self.time_mapper = None
        
        # 模型参数
        self.device = 'cpu'
        self.num_samples = 5000    # 从初始时间点采样的细胞数
        self.dt = 0.01             # SDE积分步长
        self.sigma = 0.03          # 噪声强度
        
        # 计算参数
        self.interaction_m = 1024  # interaction计算的batch size
        self.interaction_threshold = 1000
        
        # 输出设置
        self.save_dir = None       # 保存目录，默认使用exp_dir
        self.save_trajectories = True
        self.save_velocities = True

### 差值时间点来源是这个InterpolationConfig
def setup_interpolation_times(config):
    """设置插值时间点"""
    # 优先级1: 使用真实时间点（需要时间映射器）
    if config.real_times_to_interpolate is not None:
        if config.time_mapper is None:
            raise ValueError("使用真实时间插值需要设置time_mapper！")
        model_times = config.time_mapper.get_model_times(config.real_times_to_interpolate)
        return torch.tensor(model_times, dtype=torch.float32, device=config.device)
    
    # 优先级2: 直接使用指定的模型时间点
    elif config.target_times is not None:
        return torch.tensor(config.target_times, dtype=torch.float32, device=config.device)
    
    # 优先级3: 使用均匀分布的密集时间点
    else:
        return torch.linspace(config.start_time, config.end_time, 
                            config.num_frames, device=config.device)
### 细胞数量预测，有参考的growth但是没有差值的growth
### 内插情况：在已知时间范围内，使用线性插值 ； 外推情况：使用模型的生长率函数进行指数增长预测
def get_expected_cell_count_with_growth(df, t_value, x0, f_net, device='cpu'):
    """
    使用模型的生长率函数预测细胞数量
    
    Parameters:
    -----------
    df : pd.DataFrame
        原始数据
    t_value : float
        目标时间点
    x0 : torch.Tensor
        初始细胞状态
    f_net : torch.nn.Module
        训练好的模型（包含g_net）
    device : str
        计算设备
        
    Returns:
    --------
    expected_count : int
        预期的细胞数量
    """
    # 获取已知时间点的细胞数量
    known_times = sorted(df['samples'].unique())
    known_counts = [len(df[df['samples'] == t]) for t in known_times]
    
    # 如果目标时间在已知范围内，使用插值
    if known_times[0] <= t_value <= known_times[-1]:
        expected_count = int(np.interp(t_value, known_times, known_counts))
        print(f"  t={t_value:.2f}: 插值预期细胞数 {expected_count}")
        return max(expected_count, 100)
    
    # 外推情况：使用生长率模型
    print(f"  t={t_value:.2f}: 使用生长率模型外推...")
    
    # 找到最近的已知时间点
    if t_value < known_times[0]:
        ref_time = known_times[0]
        ref_count = known_counts[0]
        dt = t_value - ref_time
    else:
        ref_time = known_times[-1]
        ref_count = known_counts[-1]
        dt = t_value - ref_time
    
    # 从参考时间点采样一些细胞来估计平均生长率
    df_ref = df[df['samples'] == ref_time]
    sample_size = min(1000, len(df_ref))
    df_sample = df_ref.sample(n=sample_size, random_state=42)
    
    data_sample = torch.tensor(
        df_sample.iloc[:, 1:].values, 
        dtype=torch.float32
    ).to(device)
    
    t_tensor = torch.full(
        (sample_size, 1), 
        ref_time, 
        dtype=torch.float32, 
        device=device
    )
    
    # 计算平均生长率
    with torch.no_grad():
        _, g_values, _, _ = f_net(t_tensor, data_sample)
        mean_growth_rate = g_values.mean().item()
    
    # 使用指数增长模型：N(t) = N0 * exp(g * dt)
    expected_count = int(ref_count * np.exp(mean_growth_rate * dt))
    expected_count = max(expected_count, 100)  # 至少100个细胞
    
    print(f"  参考时间 {ref_time:.2f}, 参考细胞数 {ref_count}")
    print(f"  平均生长率 {mean_growth_rate:.4f}, dt={dt:.2f}")
    print(f"  预期细胞数: {expected_count}")
    
    return expected_count

###核心轨迹生成
def generate_trajectories_at_times_with_growth(x0, f_net, sf2m_score_model, ts_points, 
                                                initial_cell_count, config):
    """
    生成轨迹，使用 euler_sdeint_split 自动处理细胞分裂/死亡
    
    Parameters:
    -----------
    x0 : torch.Tensor
        初始状态数据
    f_net : torch.nn.Module
        训练好的模型
    sf2m_score_model : torch.nn.Module
        Score模型
    ts_points : torch.Tensor
        要插值的时间点
    initial_cell_count : int
        初始时间点的细胞数量（用于采样）
    config : InterpolationConfig
        配置参数
    """
    from DeepRUOT.interaction import euler_sdeint_split, cal_interaction
    
    class SDE(torch.nn.Module):
        noise_type = "diagonal"
        sde_type = "ito"

        def __init__(self, ode_drift, g, score, interaction, sigma=1.0):
            super().__init__()
            self.drift = ode_drift
            self.score = score
            self.sigma = sigma
            self.interaction = interaction
            self.g_net = g

        def f(self, t, y):
            z, lnw = y
            with torch.no_grad():
                drift = self.drift(t, z)
                alpha = 0.5  # 例如，设置为 0.5 将增长速度减半
                dlnw = self.g_net(t, z) * alpha
                net_forces = cal_interaction(z, lnw, self.interaction, t, 
                                            m=config.interaction_m)
            num_cells = z.shape[0]
            t_expanded = t.expand(num_cells, 1)
            score_grad = self.score.compute_gradient(t_expanded, z)
            return (drift + score_grad + net_forces, dlnw)

        def g(self, t, y):
            return torch.ones_like(y) * self.sigma

    # 1. 准备初始状态 (x0 应该已经是采样后的子集, N_init 个细胞)
    N_init = x0.shape[0]
    x0_subset = x0.to(device)
    lnw0 = torch.log(torch.ones(N_init, 1) / N_init).to(device)
    initial_state = (x0_subset, lnw0)

    # 重新初始化 SDE 对象 (使用 Cell 10 的逻辑，确保 drift 包含 score term)
    sde = SDE(f_net.v_net, 
              f_net.g_net, 
              sf2m_score_model, 
              f_net.interaction_net, 
              sigma=config.sigma)
    
    # 2. SDE 积分
    # sde_point_raw shape: [num_t, N_init, D], traj_lnw shape: [num_t, N_init, 1]
    sde_point_raw, traj_lnw = euler_sdeint_split(
        sde, 
        initial_state, 
        dt=config.dt, 
        ts=ts_points, 
        noise_std=0.0
    )
    
    all_resampled_points = []
    # all_growth_rates = []
    
    # 3. 对每个时间点进行重采样以反映细胞数量增长
    # initial_cell_count 来自外部的预期总数量
    N_target = max(1, int(initial_cell_count)) # 确保至少有一个细胞
    
    for i, t in enumerate(ts_points):
        t_value = t.item()
        current_x = sde_point_raw[i] # (N_init, D)
        current_lnw = traj_lnw[i]    # (N_init, 1)
        
        # t_tensor = torch.full((N_init, 1), t_value, device=config.device)
        # with torch.no_grad():
        #     # 使用与SDE相同的g_net计算生长率
        #     cell_growth_raw = f_net.g_net(t_tensor, current_x) * 0.5  # alpha=0.5
        
        # (b) 重采样
        current_w = torch.exp(current_lnw).squeeze().cpu().numpy()
        
        # 归一化权重作为采样概率
        p_norm = current_w / current_w.sum()
        
        # 使用替换采样 (允许克隆)
        resample_indices = np.random.choice(
            a=current_x.shape[0],      # 从 N_init 个细胞中选择
            size=N_target,             # 选择 N_target 次 (这里的 N_target 是关键)
            p=p_norm,                  # 概率由权重决定 (体现了 g_net 的作用)
            replace=True               # 允许重复选择 (即细胞分裂/克隆)
        )
        
        # 提取重采样后的细胞状态
        resampled_x = current_x[resample_indices].detach().cpu().numpy()
        all_resampled_points.append(resampled_x)
        print(f" t={t_value:.2f}: 原始 {N_init}, 重采样到 {N_target} 个细胞")
        
        # selected_growth = cell_growth_raw[resample_indices]
        # # 添加少量噪声（可选）
        # growth_noise = torch.normal(0, 0.01, size=selected_growth.shape, 
        #                             device=config.device)
        # final_growth = (selected_growth + growth_noise).cpu().numpy()
        # all_growth_rates.append(selected_growth.cpu().numpy().flatten())  # 形状 (N_target,)

    return all_resampled_points


### 速度场计算
def compute_velocity_at_timepoint(traj_np, t_value, f_net, sf2m_score_model, 
                                  dim, config):
    """
    计算特定时间点的velocity场，分开返回各分量
    
    返回:
        vel_dict: 包含所有velocity分量的字典
        all_data: 对应的数据点
        all_times: 时间标签
    """
    from DeepRUOT.interaction import cal_interaction
    
    device = config.device
    traj_np = np.asarray(traj_np, dtype=np.float32)
    n_cells = traj_np.shape[0]
    
    data_tensor = torch.tensor(traj_np, device=device, requires_grad=True)

    # ---- 1. drift term ----
    t_tensor = torch.full((n_cells, 1), float(t_value), device=device)
    
    with torch.no_grad():
        drift = f_net.v_net(t_tensor, data_tensor)
    drift_np = drift.detach().cpu().numpy()
    
    # (二) 交互项梯度 (interaction)
    lnw = torch.log(torch.ones(n_cells, 1, device=device) / n_cells)
    with torch.no_grad():
        interaction_t = cal_interaction(
            data_tensor.detach(),
            lnw,
            f_net.interaction_net,
            torch.tensor([float(t_value)], dtype=torch.float32, device=device),
            m=getattr(config, "interaction_m", 1024),
            threshold=getattr(config, "interaction_threshold", 1000),
        )
    interaction_np = interaction_t.detach().cpu().numpy()
    
    # ---- 3. score gradient term ----
    t_tensor_score = torch.full((n_cells, 1), float(t_value), device=device)

    sf2m_score_model.eval()
    for p in sf2m_score_model.parameters():
        p.requires_grad_(True)

    logp = sf2m_score_model(t_tensor_score, data_tensor)
    ones = torch.ones_like(logp)

    score_grad = torch.autograd.grad(
        outputs=logp,
        inputs=data_tensor,
        grad_outputs=ones,
        retain_graph=False,
        create_graph=False,
        only_inputs=True,
    )[0]

    score_grad_np = score_grad.detach().cpu().numpy()
    
    # 总velocity
    total_velocity = drift_np + interaction_np + score_grad_np
    
    # 打包返回
    vel_dict = {
        "drift": drift_np,
        "interaction": interaction_np,
        "score_grad": score_grad_np,
        "total": total,

        "V1_rna_intrinsic": drift_np[:, 2:],     # latent dims
        "V2_rna_interaction": interaction_np[:, 2:],
        "V3_migration_intrinsic": drift_np[:, :2],
        "V4_migration_interaction": interaction_np[:, :2],

        "coords": traj_np[:, :2],               # x1,x2
    }

    all_data = traj_np
    all_times = np.full((n_cells,), float(t_value), dtype=np.float32)
    
    return vel_dict, all_data, all_times

### 创建 scVelo 兼容的 AnnData 对象，
import anndata
def create_velocity_anndata(all_data, drift, cell_types, all_times, 
                           dim_reducer=None, use_2d=False):
    """
    创建用于scvelo可视化的AnnData对象
    
    参数:
        all_data: 细胞数据
        drift: velocity向量
        cell_types: 细胞类型标签
        all_times: 时间标签
        dim_reducer: 降维模型（如果使用）
        use_2d: 是否只使用前2维作为UMAP
    
    返回:
        adata: AnnData对象
    """
    if use_2d:
        adata = anndata.AnnData(X=all_data)
        adata.layers['Ms'] = all_data.copy()
        adata.layers['velocity'] = drift.copy()
        X_umap = all_data[:, :2]
    else:
        adata = anndata.AnnData(X=all_data[:, 2:])
        adata.layers['Ms'] = all_data.copy()
        adata.layers['velocity'] = drift.copy()
        
        if dim_reducer is not None:
            X_umap = dim_reducer.transform(all_data[:, 2:])
        else:
            X_umap = all_data[:, :2]

    adata.obsm['spatial'] = all_data[:, :2]  # 空间坐标
    adata.obsm['gene_pca'] = all_data[:, 2:]  # 基因PCA
    adata.obsm['velocity_spatial'] = drift[:, :2]  # 空间velocity
    adata.obsm['velocity_gene'] = drift[:, 2:]  # 基因velocity
    adata.obsm['X_umap'] = X_umap
    adata.obs['annotation'] = cell_types
    adata.obs['time'] = all_times
    
    return adata

### 计算 velocity graph 并进行可视化
def compute_velocity_graph(adata, frac=1.0, n_neighbors=30, n_jobs=16):
    """
    计算velocity graph并投影到UMAP
    
    参数:
        adata: AnnData对象
        frac: 采样比例
        n_neighbors: 邻居数
        n_jobs: 并行任务数
    
    返回:
        adata_sub: 处理后的AnnData子集
    """
    # 禁用tqdm进度条
    os.environ["TQDM_DISABLE"] = "1"
    
    # 采样
    if frac < 1.0:
        rng = np.random.default_rng(0)
        idx = rng.choice(adata.n_obs, size=int(adata.n_obs * frac), replace=False)
        adata_sub = adata[idx].copy()
    else:
        adata_sub = adata.copy()
    
    print(f'处理 {adata_sub.n_obs} 个细胞...')
    
    # 计算邻居图
    print('计算邻居图...')
    sc.pp.neighbors(adata_sub, n_neighbors=n_neighbors, use_rep='X')
    
    # 计算velocity图
    print('计算velocity图...')
    scv.tl.velocity_graph(adata_sub, vkey='velocity', n_jobs=n_jobs)
    
    # 投影到UMAP
    print('计算velocity embedding...')
    scv.tl.velocity_embedding(adata_sub, basis='umap', vkey='velocity')
    
    return adata_sub


In [ ]:
def interpolate_all_timepoints_fixed(original_df, f_net, sf2m_score_model, dim, 
                                    df_new, label_to_color, exp_dir, config,
                                    adata_dict, spatial_k=30, save_csv=True):
    """
    修复版插值函数 - 解决所有已知问题
    
    主要改进:
    1. 移除MLP依赖，使用纯空间KNN + 时间加权
    2. 添加空间坐标管理器，自动缩放到参考时间点范围
    3. 自动修复Y轴方向
    
    Parameters:
    -----------
    original_df : pd.DataFrame
        原始数据
    f_net, sf2m_score_model : 模型
    dim : int
        数据维度
    df_new : pd.DataFrame
        包含Annotation的数据
    label_to_color : dict
        标签到颜色映射
    exp_dir : str
        保存目录
    config : InterpolationConfig
        配置
    adata_dict : dict
        真实数据的adata字典 {time_key: adata}
    spatial_k : int
        空间KNN邻居数（降低以保留更多类型）
    save_csv : bool
        是否保存CSV
    
    Returns:
    --------
    all_adata : dict
        所有时间点的结果
    """
    from DeepRUOT.interaction import euler_sdeint_split, cal_interaction
    
    # 初始化空间坐标管理器
    coord_manager = SpatialCoordinateManager(df_new, time_column='samples')
    
    f_net = f_net.to(config.device)
    sf2m_score_model = sf2m_score_model.to(config.device)
    
    # 准备初始状态
    data = torch.tensor(original_df[original_df['samples'] == 0].values, dtype=torch.float32)
    x0 = data[:, 1:].to(config.device)
    
    ts_points = setup_interpolation_times(config)
    ts_np = ts_points.cpu().numpy()
    
    print(f"生成时间点: {ts_np}")
    
    # 找参考时间点用于Y轴修正
    real_times = sorted(adata_dict.keys())
    reference_adata = adata_dict[real_times[0]] if real_times else None
    
    # 生成所有时间点的轨迹
    print("\n=== 生成轨迹（使用生长率模型） ===")
    all_sde_points = []
    all_growth_rates = []
    
    for t in ts_np:
        expected_count = get_expected_cell_count_with_growth(
            original_df, float(t), x0, f_net, config.device
        )
        
        ts_single = torch.tensor([t], dtype=torch.float32, device=config.device)
        sde_frames = generate_trajectories_at_times_with_growth(
            x0, f_net, sf2m_score_model, ts_single, 
            initial_cell_count=expected_count,
            config=config
        )
        all_sde_points.append(sde_frames[0])
        # all_growth_rates.append(cell_growth_rates[0])  # 保存生长率
        print(f"  t={t:.2f}: 生成 {sde_frames[0].shape[0]} 个细胞")
    
    # 处理每个时间点
    print("\n=== 处理每个时间点 ===")
    all_adata = {}
    
    for i, t in enumerate(ts_np):
        traj_data = all_sde_points[i]
        # growth_rates = all_growth_rates[i]
        print(f"\n处理时间点 {i+1}/{len(ts_np)}: t={t:.2f}")
        
        # ===修复1: 空间坐标缩放===
        original_coords = traj_data[:, :2].copy()
        scaled_coords = coord_manager.rescale_to_reference(original_coords, float(t))
        traj_data[:, :2] = scaled_coords  # 更新空间坐标
        
        # 计算velocity
        vel_dict, all_data, all_times = compute_velocity_from_sdepoint(
            traj_data, float(t), f_net, sf2m_score_model, dim, config
        )
        
        
        # ===修复2: 改进的细胞类型预测（移除MLP）===
        cell_types = predict_cell_types_adaptive_knn(
            traj_data=traj_data,
            t_value=float(t),
            df_new=df_new,
            label_to_color=label_to_color,
            spatial_k=spatial_k,
            time_weight_decay=0.5
        )
        
        ###=========输出细胞的growth===========
        data_all = torch.tensor(
            traj_data, 
            dtype=torch.float32
        )
        t_all_tensor = torch.full(
            (data_all.shape[0],1),
            float(t), 
            dtype=torch.float32, 
            device=device
        )
        with torch.no_grad():
            growth_values_raw = f_net.g_net(t_all_tensor, data_all)
            growth_values = growth_values_raw * 0.5
        
        
        # 创建adata
        adata = create_velocity_anndata(
            all_data, vel_dict['total'], cell_types, all_times, use_2d=True
        )
        
        # # ===修复3: Y轴方向修正=== 
        # adata = fix_y_axis_orientation(adata, reference_adata=reference_adata) 
        
        # 保存所有信息
        growth_values_np = growth_values.cpu().numpy().flatten()
        
        adata.obs['growth_rate'] = growth_values_np
        adata.obs['annotation'] = cell_types
        adata.obsm['V_drift'] = vel_dict['drift']
        adata.obsm['V_interaction'] = vel_dict['interaction']
        adata.obsm['V_score'] = vel_dict['score_grad']
        adata.obsm['V_total'] = vel_dict['total']
        adata.obsm['V1_rna_intrinsic'] = vel_dict['V1_rna_intrinsic']
        adata.obsm['V2_rna_interaction'] = vel_dict['V2_rna_interaction']
        adata.obsm['V3_migration_intrinsic'] = vel_dict['V3_migration_intrinsic']
        adata.obsm['V4_migration_interaction'] = vel_dict['V4_migration_interaction']
        
        adata.obs['cell_colors'] = [label_to_color.get(lbl, '#808080') for lbl in cell_types]
        adata.uns['time'] = float(t)
        adata.uns['actual_count'] = traj_data.shape[0]
        adata.uns['prediction_method'] = 'adaptive_spatial_knn'
        adata.uns['spatial_scaling'] = 'auto_reference'
        adata.uns['y_axis'] = 'corrected'
        
        # 保存
        t_str = f"{float(t):.3f}".replace('.', 'p').replace('-', 'n')
        save_path = os.path.join(exp_dir, f'adata_t{t_str}.h5ad')
        adata.write(save_path)
        print(f"  已保存: {save_path}")
        
        all_adata[float(t)] = adata
        
        if save_csv:
            save_sdeframe_to_csv(traj_data, float(t), 
                               os.path.join(exp_dir, 'interpolated_csv'), dim)
        
        print(f"完成 t={t:.3f}: {len(np.unique(cell_types))} 个细胞类型")
    
    return all_adata

In [ ]:
### 速度场计算模块
def compute_velocity_from_sdepoint(sde_point_data, t_value, f_net, 
                                   sf2m_score_model, dim, config):
    """
    从sde_point数据计算velocity场，分开返回各分量
    
    Parameters:
    -----------
    sde_point_data : np.ndarray
        单个时间点的sde_point数据，shape (n_cells, dim)
    t_value : float
        时间点
    f_net, sf2m_score_model : 模型
    dim : int
        数据维度
    config : 配置对象
    
    Returns:
    --------
    vel_dict : dict
        包含所有velocity分量的字典
    all_data : np.ndarray
        对应的数据点
    all_times : np.ndarray
        时间标签
    """
    from DeepRUOT.interaction import cal_interaction
    
    device = config.device
    traj_np = np.asarray(sde_point_data, dtype=np.float32)
    n_cells = traj_np.shape[0]
    
    data_tensor = torch.tensor(traj_np, device=device, requires_grad=True)
    t_tensor = torch.full((n_cells, 1), float(t_value), device=device)
    
    # 1. Drift term
    with torch.no_grad():
        drift = f_net.v_net(t_tensor, data_tensor)
    drift_np = drift.detach().cpu().numpy()
    
    # 2. Interaction term
    lnw = torch.log(torch.ones(n_cells, 1, device=device) / n_cells)
    with torch.no_grad():
        interaction_t = cal_interaction(
            data_tensor.detach(),
            lnw,
            f_net.interaction_net,
            torch.tensor([float(t_value)], dtype=torch.float32, device=device),
            m=getattr(config, "interaction_m", 1024),
            threshold=getattr(config, "interaction_threshold", 1000),
        )
    interaction_np = interaction_t.detach().cpu().numpy()
    
    # 3. Score gradient term
    sf2m_score_model.eval()
    for p in sf2m_score_model.parameters():
        p.requires_grad_(True)

    logp = sf2m_score_model(t_tensor, data_tensor)
    ones = torch.ones_like(logp)

    score_grad = torch.autograd.grad(
        outputs=logp,
        inputs=data_tensor,
        grad_outputs=ones,
        retain_graph=False,
        create_graph=False,
        only_inputs=True,
    )[0]

    score_grad_np = score_grad.detach().cpu().numpy()
    
    # 总velocity
    total_velocity = drift_np + interaction_np + score_grad_np
    
    # 打包返回
    vel_dict = {
        "drift": drift_np,
        "interaction": interaction_np,
        "score_grad": score_grad_np,
        "total": total_velocity,
        "V1_rna_intrinsic": drift_np[:, 2:],          # latent dims
        "V2_rna_interaction": interaction_np[:, 2:],
        "V3_migration_intrinsic": drift_np[:, :2],
        "V4_migration_interaction": interaction_np[:, :2],
        "coords": traj_np[:, :2],                     # x1,x2
    }

    all_data = traj_np
    all_times = np.full((n_cells,), float(t_value), dtype=np.float32)
    
    return vel_dict, all_data, all_times

### 将SDE模拟生成的细胞状态数据保存为CSV格式文件，以便后续分析和可视化。
def save_sdeframe_to_csv(sde_point_data, t_value, save_dir, dim):
    """
    保存单个时间点的sde_point数据为CSV
    
    Parameters:
    -----------
    sde_point_data : np.ndarray
        shape (n_cells, dim)
    t_value : float
        时间点
    save_dir : str
        保存目录
    dim : int
        数据维度
    
    Returns:
    --------
    csv_path : str
        保存的CSV文件路径
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # 构建DataFrame
    column_names = ['samples'] + [f'x{i}' for i in range(1, dim + 1)]
    n_cells = sde_point_data.shape[0]
    
    data_dict = {'samples': np.full(n_cells, t_value)}
    for i in range(dim):
        data_dict[f'x{i+1}'] = sde_point_data[:, i]
    
    df_sde = pd.DataFrame(data_dict)
    
    # 保存
    t_str = f"{t_value:.3f}".replace('.', 'p').replace('-', 'n')
    csv_path = os.path.join(save_dir, f'sde_point_t{t_str}.csv')
    df_sde.to_csv(csv_path, index=False)
    
    return csv_path

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
def predict_cell_types_adaptive_knn(traj_data, t_value, df_new, label_to_color,
                                    spatial_k=20, time_weight_decay=0.5):
    """
    基于最近时间点的自适应KNN预测，移除MLP依赖
    
    Parameters:
    -----------
    traj_data : np.ndarray
        插值数据 (n_cells, dim)，前两维是空间坐标
    t_value : float
        当前时间点
    df_new : pd.DataFrame
        包含真实时间点数据，必须有 ['samples', 'x1', 'x2', 'Annotation']
    label_to_color : dict
        标签到颜色的映射
    spatial_k : int
        空间KNN邻居数（降低到30以保留更多类型）
    time_weight_decay : float
        时间距离衰减系数（越大则远时间点影响越小）
    
    Returns:
    --------
    predicted_labels : np.ndarray
        预测的细胞类型
    """
    
    available_times = sorted(df_new['samples'].unique())
    n_cells = traj_data.shape[0]
    coords = traj_data[:, :2]  # 空间坐标
    
    # 找到最近的1-2个时间点
    if t_value <= available_times[0]:
        # 外推到最早之前
        nearest_times = [available_times[0]]
        weights = [1.0]
    elif t_value >= available_times[-1]:
        # 外推到最晚之后
        nearest_times = [available_times[-1]]
        weights = [1.0]
    else:
        # 内插：找前后两个时间点
        for i in range(len(available_times) - 1):
            if available_times[i] <= t_value <= available_times[i + 1]:
                t_before = available_times[i]
                t_after = available_times[i + 1]
                
                # 计算时间距离权重（使用指数衰减）
                dist_before = abs(t_value - t_before)
                dist_after = abs(t_value - t_after)
                
                # 指数衰减权重
                w_before = np.exp(-time_weight_decay * dist_before)
                w_after = np.exp(-time_weight_decay * dist_after)
                total = w_before + w_after
                
                nearest_times = [t_before, t_after]
                weights = [w_before / total, w_after / total]
                break
    
    print(f"  预测 t={t_value:.2f}: 使用参考时间点 {nearest_times} (权重: {[f'{w:.3f}' for w in weights]})")
    
    # 对每个参考时间点进行KNN预测
    all_predictions = []
    all_classes = df_new['Annotation'].unique()
    
    for ref_time, weight in zip(nearest_times, weights):
        df_ref = df_new[df_new['samples'] == ref_time]
        ref_coords = df_ref[['x1', 'x2']].values
        ref_labels = df_ref['Annotation'].values
        
        # 动态调整k值（避免k大于样本数）
        effective_k = min(spatial_k, max(5, len(ref_coords) // 10))
        
        # 训练KNN（仅使用空间坐标）
        knn = KNeighborsClassifier(n_neighbors=effective_k, weights='distance')
        knn.fit(ref_coords, ref_labels)
        
        # 获取概率分布
        knn_probs = knn.predict_proba(coords)
        
        # 对齐到全部类别
        knn_classes = knn.classes_
        aligned_probs = np.zeros((n_cells, len(all_classes)))
        
        for i, cls in enumerate(knn_classes):
            cls_idx = np.where(all_classes == cls)[0]
            if len(cls_idx) > 0:
                aligned_probs[:, cls_idx[0]] = knn_probs[:, i]
        
        all_predictions.append((aligned_probs, weight))
    
    # 加权平均
    final_probs = np.zeros((n_cells, len(all_classes)))
    for probs, weight in all_predictions:
        final_probs += probs * weight
    
    # 最终预测（添加少数类保护）
    final_labels_idx = final_probs.argmax(axis=1)
    
    # 少数类保护：如果某个细胞的最高概率与次高概率差距小于阈值，且次高是少数类，则优先保留
    prob_margin = 0.15  # 概率差阈值
    sorted_probs = np.sort(final_probs, axis=1)
    max_prob = sorted_probs[:, -1]
    second_max_prob = sorted_probs[:, -2]
    
    uncertain_mask = (max_prob - second_max_prob) < prob_margin
    
    if uncertain_mask.sum() > 0:
        # 对不确定的细胞，检查次高概率是否对应少数类
        second_max_idx = np.argsort(final_probs, axis=1)[:, -2]
        
        # 计算各类别在参考时间点的丰度
        type_counts = df_new[df_new['samples'].isin(nearest_times)]['Annotation'].value_counts()
        rare_types = type_counts[type_counts < type_counts.quantile(0.25)].index
        
        for i in np.where(uncertain_mask)[0]:
            second_type = all_classes[second_max_idx[i]]
            if second_type in rare_types:
                final_labels_idx[i] = second_max_idx[i]
    
    final_labels = all_classes[final_labels_idx]
    
    unique_types = np.unique(final_labels)
    print(f"  预测完成: {len(unique_types)} 个细胞类型，样本数: {len(final_labels)}")
    print(f"  类型分布: {dict(zip(*np.unique(final_labels, return_counts=True)))}")
    
    return final_labels

In [ ]:
def load_adata_timepoint(exp_dir, t_value):
    """读取单个时间点的adata"""
    import scanpy as sc
    t_str = f"{t_value:.3f}".replace('.', 'p').replace('-', 'n')
    path = os.path.join(exp_dir, f'adata_t{t_str}.h5ad')
    adata = sc.read_h5ad(path)
    print(f"读取 t={t_value:.3f}, {adata.n_obs} 个细胞")
    return adata

def plot_timepoint(exp_dir, t_value, plot_type='all'):
    """
    读取并画单个时间点
    plot_type: 'fingerprint', 'celltype', 'components', 'all'
    """
    adata = load_adata_timepoint(exp_dir, t_value)
    coords = adata.obsm['X_umap']
    t_str = f"{t_value:.2f}"
    
    # 从adata提取所有velocity分量
    V1 = adata.obsm['V1_rna_intrinsic']
    V2 = adata.obsm['V2_rna_interaction']
    V3 = adata.obsm['V3_migration_intrinsic']
    V4 = adata.obsm['V4_migration_interaction']
    
    if plot_type in ['fingerprint', 'all']:
        rgb_colors, (R, G, B, Y) = create_rgb_fingerprint(V1, V2, V3, V4)
        
        fig, ax = plt.subplots(figsize=(20, 20), facecolor='black')
        ax.set_facecolor('black')
        ax.scatter(coords[:, 0], coords[:, 1], c=rgb_colors, s=8, alpha=0.8)
        ax.set_title(f'Velocity Fingerprint (t={t_str})', color='white', fontsize=14)
        ax.set_aspect('equal')
        ax.axis('off')
        plt.savefig(os.path.join(exp_dir, f'fingerprint_t{t_str}.png'), 
                    dpi=300, facecolor='black', bbox_inches='tight')
        plt.show()
    
    if plot_type in ['celltype', 'all']:
        fig, ax = plt.subplots(figsize=(20, 20))
        # 获取唯一的cell type和对应颜色
        cell_types = adata.obs['annotation'].values
        cell_colors = adata.obs['cell_colors'].values
        unique_types = np.unique(cell_types)
        
        # 为每个cell type单独画，这样可以加图例
        for ct in unique_types:
            mask = cell_types == ct
            color = cell_colors[mask][0]  # 该类型的颜色
            ax.scatter(coords[mask, 0], coords[mask, 1], c=color, s=8, alpha=0.8, label=ct)
        
        ax.set_title(f'Cell Types (t={t_str})', fontsize=14)
        ax.set_aspect('equal')
        
        # 添加图例
        ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), 
                  fontsize=8, markerscale=2, frameon=False)
        
        plt.tight_layout()
        plt.savefig(os.path.join(exp_dir, f'celltype_t{t_str}.png'), dpi=300, bbox_inches='tight')
        plt.show()
        
    if plot_type in ['components', 'all']:
        rgb_colors, (R, G, B, Y) = create_rgb_fingerprint(V1, V2, V3, V4)
        
        fig, axes = plt.subplots(2, 2, figsize=(12, 12))
        channel_data = [
            (R, 'Cell Intrinsic', 'Reds'),
            (G, 'Cell Interaction', 'Greens'),
            (B, 'Migration Intrinsic', 'Blues'),
            (Y, 'Migration Interaction', 'YlOrRd')
        ]
        
        for ax, (channel, title, cmap) in zip(axes.flat, channel_data):
            scatter = ax.scatter(coords[:, 0], coords[:, 1], c=channel, 
                                s=15, alpha=0.8, cmap=cmap, vmin=0, vmax=1)
            ax.set_title(title, fontsize=14, fontweight='bold')
            ax.set_aspect('equal')
            plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
        
        fig.suptitle(f'Velocity Components (t={t_str})', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(exp_dir, f'components_t{t_str}.png'), dpi=300, bbox_inches='tight')
        plt.show()
    
    return adata

class SpatialCoordinateManager:
    """管理空间坐标的缩放和反缩放"""
    
    def __init__(self, df_original, time_column='samples'):
        """
        初始化坐标管理器
        
        Parameters:
        -----------
        df_original : pd.DataFrame
            原始数据，包含各时间点的真实空间坐标
        time_column : str
            时间列名
        """
        self.time_column = time_column
        self.scaling_params = {}
        
        # 为每个时间点记录缩放参数
        for t in df_original[time_column].unique():
            df_t = df_original[df_original[time_column] == t]
            x1_min, x1_max = df_t['x1'].min(), df_t['x1'].max()
            x2_min, x2_max = df_t['x2'].min(), df_t['x2'].max()
            
            self.scaling_params[float(t)] = {
                'x1_min': x1_min,
                'x1_max': x1_max,
                'x2_min': x2_min,
                'x2_max': x2_max,
                'x1_range': x1_max - x1_min,
                'x2_range': x2_max - x2_min
            }
        
        print("空间坐标管理器初始化完成:")
        for t, params in self.scaling_params.items():
            print(f"  t={t}: X1=[{params['x1_min']:.2f}, {params['x1_max']:.2f}], "
                  f"X2=[{params['x2_min']:.2f}, {params['x2_max']:.2f}]")
    
    def get_reference_time(self, t_value):
        """获取最近的参考时间点"""
        times = sorted(self.scaling_params.keys())
        if t_value <= times[0]:
            return times[0]
        elif t_value >= times[-1]:
            return times[-1]
        else:
            # 找最近的时间点
            idx = np.searchsorted(times, t_value)
            if idx == 0:
                return times[0]
            elif idx >= len(times):
                return times[-1]
            else:
                # 返回距离更近的那个
                dist_before = abs(t_value - times[idx - 1])
                dist_after = abs(t_value - times[idx])
                return times[idx - 1] if dist_before < dist_after else times[idx]
    
    def rescale_to_reference(self, coords, t_value, margin=0.05):
        """
        将插值坐标缩放到参考时间点的空间范围
        
        Parameters:
        -----------
        coords : np.ndarray
            归一化后的坐标 (n, 2)
        t_value : float
            当前时间点
        margin : float
            边缘留白比例
        
        Returns:
        --------
        scaled_coords : np.ndarray
            缩放到真实空间的坐标
        """
        ref_time = self.get_reference_time(t_value)
        params = self.scaling_params[ref_time]
        
        # 当前坐标的范围
        x1_min_curr, x1_max_curr = coords[:, 0].min(), coords[:, 0].max()
        x2_min_curr, x2_max_curr = coords[:, 1].min(), coords[:, 1].max()
        
        # 归一化到[0, 1]
        if x1_max_curr > x1_min_curr:
            norm_x1 = (coords[:, 0] - x1_min_curr) / (x1_max_curr - x1_min_curr)
        else:
            norm_x1 = np.ones_like(coords[:, 0]) * 0.5
        
        if x2_max_curr > x2_min_curr:
            norm_x2 = (coords[:, 1] - x2_min_curr) / (x2_max_curr - x2_min_curr)
        else:
            norm_x2 = np.ones_like(coords[:, 1]) * 0.5
        
        # 计算目标范围（带边距）
        x1_range_target = params['x1_range'] * (1 - 2 * margin)
        x2_range_target = params['x2_range'] * (1 - 2 * margin)
        
        x1_min_target = params['x1_min'] + params['x1_range'] * margin
        x2_min_target = params['x2_min'] + params['x2_range'] * margin
        
        # 缩放到目标范围
        scaled_x1 = x1_min_target + norm_x1 * x1_range_target
        scaled_x2 = x2_min_target + norm_x2 * x2_range_target
        
        scaled_coords = np.column_stack([scaled_x1, scaled_x2])
        
        print(f"  坐标缩放 t={t_value:.2f} (参考t={ref_time}):")
        print(f"    原始范围: X1=[{x1_min_curr:.3f}, {x1_max_curr:.3f}], X2=[{x2_min_curr:.3f}, {x2_max_curr:.3f}]")
        print(f"    目标范围: X1=[{x1_min_target:.2f}, {x1_min_target+x1_range_target:.2f}], "
              f"X2=[{x2_min_target:.2f}, {x2_min_target+x2_range_target:.2f}]")
        print(f"    缩放后范围: X1=[{scaled_x1.min():.2f}, {scaled_x1.max():.2f}], "
              f"X2=[{scaled_x2.min():.2f}, {scaled_x2.max():.2f}]")
        
        return scaled_coords

def fix_y_axis_orientation(adata, reference_adata=None):
    """
    修复Y轴方向，使其与参考数据一致
    
    Parameters:
    -----------
    adata : AnnData
        需要修正的数据
    reference_adata : AnnData, optional
        参考数据，如果提供则自动检测方向
    
    Returns:
    --------
    adata : AnnData
        修正后的数据
    """
    spatial = adata.obsm['spatial'].copy()
    
    if reference_adata is not None:
        # 自动检测：比较Y坐标的方向
        ref_y_mean = reference_adata.obsm['spatial'][:, 1].mean()
        curr_y_mean = spatial[:, 1].mean()
        
        # 如果参考数据的Y均值更大，说明需要翻转
        if ref_y_mean > curr_y_mean * 1.5:  # 1.5是容差系数
            spatial[:, 1] = spatial[:, 1].max() - spatial[:, 1] + spatial[:, 1].min()
            print(f"  Y轴已翻转（参考Y均值={ref_y_mean:.2f}, 当前Y均值={curr_y_mean:.2f}）")
    else:
        # 手动翻转（总是翻转，因为SDE积分通常使用数学坐标系）
        y_max = spatial[:, 1].max()
        y_min = spatial[:, 1].min()
        spatial[:, 1] = y_max - spatial[:, 1] + y_min
        print(f"  Y轴已翻转（范围: [{y_min:.2f}, {y_max:.2f}]）")
    
    adata.obsm['spatial'] = spatial
    
    # 同时更新X_umap如果存在
    if 'X_umap' in adata.obsm and adata.obsm['X_umap'].shape[1] >= 2:
        adata.obsm['X_umap'][:, :2] = spatial.copy()
    
    return adata

In [ ]:
time_mapper = TimeMapper(
    real_times=[12.5, 13.5, 14.5, 15.5],  # 已有的批次时间
    model_times=[0, 1, 2, 3]               # 对应的模型时间
)

In [ ]:
# 查看映射关系
time_mapper.print_mapping()

In [ ]:
# 配置并运行插值
config = InterpolationConfig()
config.time_mapper = time_mapper
# 模型时间外推

# 使用均匀分布的100个时间点
config.start_time = 1.0      # 起始时间（模型时间）
config.end_time = 2.0        # 结束时间（模型时间）
config.num_frames = 10     # 总帧数（10个插值点）

# 注意：这里要设置为None，让代码使用优先级3
config.real_times_to_interpolate = None
config.target_times = None

config.sigma = 0.03  # 噪声强度，影响细胞分裂/死亡
config.dt = 0.01

In [ ]:
adata_dict

selected_timepoints = ["E12.5", "E13.5", "E14.5", "E15.5"]

adata_subdict = {tp: adata_dict[tp] for tp in selected_timepoints}

In [ ]:
all_adata = interpolate_all_timepoints_fixed(
    original_df=df,
    f_net=f_net,
    sf2m_score_model=sf2m_score_model,
    dim=dim,
    df_new=df_new,
    label_to_color=label_to_color,
    exp_dir=exp_dir,
    config=config,
    adata_dict=adata_subdict,  # 真实时间点的adata
    spatial_k=30,  # 降低K值以保留更多细胞类型
    save_csv=False,
)

In [ ]:
all_adata

In [ ]:
import scanpy as sc
import os
import re

def load_adata_by_time_range(adata_dir='/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/', start_time=None, end_time=None):
    """
    按时间范围加载adata文件
    
    Parameters:
    -----------
    adata_dir : str
        文件夹路径
    start_time : float or None
        起始时间，None表示从最早开始
    end_time : float or None
        结束时间，None表示到最后
    
    Returns:
    --------
    all_adata : dict
        {时间点: adata} 字典，只包含指定范围内的时间点
    """
    all_adata = {}
    
    for filename in os.listdir(adata_dir):
        if filename.endswith('.h5ad') and filename.startswith('adata_t'):
            # 从文件名提取时间
            match = re.search(r'adata_t([0-9]+p[0-9]+)\.h5ad', filename)
            if match:
                time_str = match.group(1)  # 例如 "1p000"
                time_str_float = time_str.replace('p', '.')
                
                try:
                    time_point = float(time_str_float)
                    
                    # 检查时间是否在指定范围内
                    if start_time is not None and time_point < start_time:
                        continue
                    if end_time is not None and time_point > end_time:
                        continue
                    
                    file_path = os.path.join(adata_dir, filename)
                    adata = sc.read_h5ad(file_path)
                    all_adata[time_point] = adata
                    
                    print(f"加载: t={time_point:.3f}")
                    
                except (ValueError, Exception) as e:
                    print(f"跳过 {filename}: {e}")
    
    # 按时间排序
    all_adata = dict(sorted(all_adata.items()))
    
    print(f"\n加载完成！")
    print(f"时间范围: {start_time} - {end_time}")
    print(f"实际加载: {list(all_adata.keys())}")
    print(f"总数: {len(all_adata)} 个时间点")
    
    return all_adata

In [ ]:
#all_adata = load_adata_by_time_range(start_time=1.0, end_time=2.0)

## Growth

In [ ]:
def extract_cell_growth_data(all_adata, target_cell_type):
    """
    Extract growth rate data for ANY cell type from all time points
    
    Parameters:
    -----------
    all_adata : dict
        Dictionary of AnnData objects by time
    target_cell_type : str or list
        Target cell type name(s) to analyze
        Can be a single string or list of strings for multiple patterns
    
    Returns:
    --------
    cell_df : pd.DataFrame
        DataFrame containing growth rate data for the target cell type
    """
    import pandas as pd
    import numpy as np
    
    cell_data = []  # Store all cell growth rate data
    
    # Convert target_cell_type to list if it's a string
    if isinstance(target_cell_type, str):
        target_patterns = [target_cell_type]
    else:
        target_patterns = target_cell_type
    
    print(f"=== Extracting Growth Data for: {target_patterns} ===")
    
    for t, adata in all_adata.items():
        # Check if necessary columns exist
        if 'annotation' not in adata.obs.columns:
            print(f"Time {t:.2f}: No 'annotation' column found")
            continue
        
        if 'growth_rate' not in adata.obs.columns:
            print(f"Time {t:.2f}: No 'growth_rate' column found")
            continue
        
        # Identify target cells - flexible matching
        cell_mask = np.zeros(len(adata), dtype=bool)
        
        for pattern in target_patterns:
            # Try different matching strategies
            mask1 = adata.obs['annotation'].astype(str).str.contains(
                pattern, case=False, na=False, regex=False
            )
            
            # Also try exact match
            mask2 = adata.obs['annotation'].astype(str).str.lower() == pattern.lower()
            
            # Combine masks
            pattern_mask = mask1 | mask2
            cell_mask = cell_mask | pattern_mask
        
        # Get target cells
        cell_indices = np.where(cell_mask)[0]
        
        if len(cell_indices) > 0:
            # Extract growth rates
            cell_growth_rates = adata.obs['growth_rate'].values[cell_indices]
            
            # Also get the actual cell type labels for verification
            cell_labels = adata.obs['annotation'].values[cell_indices]
            
            # Create record for each cell
            for growth_rate, label in zip(cell_growth_rates, cell_labels):
                cell_data.append({
                    'time': t,
                    'growth_rate': float(growth_rate),
                    'cell_type_label': label,  # Actual label found
                    'target_pattern': ', '.join(target_patterns),
                    'cell_id': f"t{t:.2f}_{len(cell_data)}"  # Unique ID
                })
            
            # Find unique labels for this time point
            unique_labels = np.unique(cell_labels)
            print(f"Time {t:.2f}: Found {len(cell_indices)} cells")
            print(f"  Matching labels: {list(unique_labels)[:5]}...")
            print(f"  Growth rate stats: mean={np.mean(cell_growth_rates):.4f}, "
                  f"std={np.std(cell_growth_rates):.4f}")
        else:
            # Check what cell types are available at this time
            available_types = adata.obs['annotation'].unique()[:5]
            print(f"Time {t:.2f}: No '{target_patterns}' cells found")
            print(f"  Available types: {list(available_types)}...")
    
    # Convert to DataFrame
    if cell_data:
        cell_df = pd.DataFrame(cell_data)
        
        print(f"\n=== EXTRACTION SUMMARY ===")
        print(f"Target cell type(s): {target_patterns}")
        print(f"Total cells found: {len(cell_df)}")
        if len(cell_df) > 0:
            print(f"Time range: {cell_df['time'].min():.2f} - {cell_df['time'].max():.2f}")
            print(f"Growth rate range: [{cell_df['growth_rate'].min():.4f}, "
                  f"{cell_df['growth_rate'].max():.4f}]")
            
            # Show what labels were actually found
            actual_labels = cell_df['cell_type_label'].unique()
            print(f"\nActual labels found ({len(actual_labels)} unique):")
            for label in actual_labels[:10]:  # Show first 10
                count = len(cell_df[cell_df['cell_type_label'] == label])
                print(f"  - '{label}': {count} cells")
            if len(actual_labels) > 10:
                print(f"  ... and {len(actual_labels) - 10} more")
    else:
        print("\n⚠ NO DATA FOUND!")
        print("Check if the cell type name is correct.")
        cell_df = pd.DataFrame()  # Return empty DataFrame
    
    return cell_df


def plot_multiple_cell_curves(all_adata, cell_type_list, save_path=None, 
                             y_limits=(0.0, 0.1), line_width=2.5):
    """
    Plot smooth curves for multiple cell types on the same plot
    
    Parameters:
    -----------
    all_adata : dict
        Dictionary of AnnData objects by time
    cell_type_list : list
        List of cell type names to plot
    save_path : str, optional
        Path to save the figure
    y_limits : tuple, optional
        Y-axis limits (min, max)
    line_width : float, optional
        Width of the curves
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from scipy.interpolate import make_interp_spline
    import pandas as pd
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    
    # Define color palette
    colors = [
        '#2E86AB',  # Blue
        '#A23B72',  # Purple
        '#F18F01',  # Orange
        '#73AB84',  # Green
        '#C14953',  # Red
        '#6D435A',  # Dark Purple
        '#00A6A6',  # Teal
        '#F0B67F',  # Light Orange
        '#7D82B8',  # Lavender
        '#D56F3E',  # Brown
    ]
    
    # Store all curves for legend
    curve_handles = []
    curve_labels = []
    
    # Plot each cell type
    for i, cell_type in enumerate(cell_type_list):
        # Extract data for this cell type
        cell_df = extract_cell_growth_data(all_adata, cell_type)
        
        if len(cell_df) == 0:
            print(f"⚠ No data found for: {cell_type}")
            continue
        
        # Prepare data
        x = cell_df['time'].values
        y = cell_df['growth_rate'].values
        
        # Sort
        sort_idx = np.argsort(x)
        x_sorted = x[sort_idx]
        y_sorted = y[sort_idx]
        
        # Create smooth curve
        if len(x_sorted) > 3:
            # Smooth x values
            x_smooth = np.linspace(x_sorted.min(), x_sorted.max(), 500)
            
            # Cubic spline
            try:
                spline = make_interp_spline(x_sorted, y_sorted, k=3)
                y_smooth = spline(x_smooth)
            except:
                # Polynomial fallback
                degree = min(3, len(x_sorted) - 1)
                coeffs = np.polyfit(x_sorted, y_sorted, degree)
                poly = np.poly1d(coeffs)
                y_smooth = poly(x_smooth)
                x_smooth = x_smooth  # Already defined
            
            # Plot the curve
            color = colors[i % len(colors)]
            line, = ax.plot(x_smooth, y_smooth, '-', 
                           color=color,
                           linewidth=line_width,
                           alpha=0.9,
                           label=cell_type)
            
            curve_handles.append(line)
            curve_labels.append(cell_type)
            
            print(f"✓ Plotted: {cell_type} ({len(cell_df)} cells)")
        else:
            print(f"⚠ Not enough data points for smoothing: {cell_type}")
    
    # Set y-axis limits
    ax.set_ylim(y_limits)
    
    # Add horizontal line at y=0
    ax.axhline(y=0, color='black', linestyle='-', 
              linewidth=0.8, alpha=0.3, zorder=1)
    
    # Add subtle grid
    ax.grid(True, axis='y', alpha=0.15, linestyle='--', linewidth=0.5)
    ax.grid(True, axis='x', alpha=0.1, linestyle=':', linewidth=0.5)
    
    # Labels
    ax.set_xlabel('Time', fontsize=12, fontweight='bold')
    ax.set_ylabel('Growth Rate', fontsize=12, fontweight='bold')
    
    # Title
    ax.set_title('Multiple Cell Types: Growth Rate Trends', 
                fontsize=14, fontweight='bold', pad=20)
    
    # Legend
    ax.legend(handles=curve_handles, labels=curve_labels,
             loc='best', fontsize=10, framealpha=0.9,
             fancybox=True, shadow=True)
    
    # Adjust layout
    plt.tight_layout()
    
    # Save figure
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        print(f"\nMulti-cell plot saved to: {save_path}")
    
    plt.show()

In [ ]:
def plot_multiple_cell_curves(all_adata, cell_type_list, save_path=None, 
                             y_limits=(0.0, 0.1), line_width=2.5):
    """
    Plot smooth curves for multiple cell types on the same plot
    
    Parameters:
    -----------
    all_adata : dict
        Dictionary of AnnData objects by time
    cell_type_list : list
        List of cell type names to plot
    save_path : str, optional
        Path to save the figure
    y_limits : tuple, optional
        Y-axis limits (min, max)
    line_width : float, optional
        Width of the curves
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from scipy.interpolate import make_interp_spline
    import pandas as pd
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    
    # Define color palette
    colors = [
        '#2E86AB',  # Blue
        '#A23B72',  # Purple
        '#F18F01',  # Orange
        '#73AB84',  # Green
        '#C14953',  # Red
        '#6D435A',  # Dark Purple
        '#00A6A6',  # Teal
        '#F0B67F',  # Light Orange
        '#7D82B8',  # Lavender
        '#D56F3E',  # Brown
    ]
    
    # Store all curves for legend
    curve_handles = []
    curve_labels = []
    
    # Plot each cell type
    for i, cell_type in enumerate(cell_type_list):
        # Extract data for this cell type
        cell_df = extract_cell_growth_data(all_adata, cell_type)
        
        if len(cell_df) == 0:
            print(f"⚠ No data found for: {cell_type}")
            continue
        
        # Prepare data
        x = cell_df['time'].values
        y = cell_df['growth_rate'].values
        
        # Sort
        sort_idx = np.argsort(x)
        x_sorted = x[sort_idx]
        y_sorted = y[sort_idx]
        
        # Create smooth curve
        if len(x_sorted) > 3:
            # Smooth x values
            x_smooth = np.linspace(x_sorted.min(), x_sorted.max(), 500)
            
            # Cubic spline
            try:
                spline = make_interp_spline(x_sorted, y_sorted, k=3)
                y_smooth = spline(x_smooth)
            except:
                # Polynomial fallback
                degree = min(3, len(x_sorted) - 1)
                coeffs = np.polyfit(x_sorted, y_sorted, degree)
                poly = np.poly1d(coeffs)
                y_smooth = poly(x_smooth)
                x_smooth = x_smooth  # Already defined
            
            # Plot the curve
            color = colors[i % len(colors)]
            line, = ax.plot(x_smooth, y_smooth, '-', 
                           color=color,
                           linewidth=line_width,
                           alpha=0.9,
                           label=cell_type)
            
            curve_handles.append(line)
            curve_labels.append(cell_type)
            
            print(f"✓ Plotted: {cell_type} ({len(cell_df)} cells)")
        else:
            print(f"⚠ Not enough data points for smoothing: {cell_type}")
    
    # Set y-axis limits
    ax.set_ylim(y_limits)
    
    # Add horizontal line at y=0
    ax.axhline(y=0, color='black', linestyle='-', 
              linewidth=0.8, alpha=0.3, zorder=1)
    
    # Add subtle grid
    ax.grid(True, axis='y', alpha=0.15, linestyle='--', linewidth=0.5)
    ax.grid(True, axis='x', alpha=0.1, linestyle=':', linewidth=0.5)
    
    # Labels
    ax.set_xlabel('Time', fontsize=12, fontweight='bold')
    ax.set_ylabel('Growth Rate', fontsize=12, fontweight='bold')
    
    # Title
    ax.set_title('Multiple Cell Types: Growth Rate Trends', 
                fontsize=14, fontweight='bold', pad=20)
    
    # Legend
    ax.legend(handles=curve_handles, labels=curve_labels,
             loc='best', fontsize=10, framealpha=0.9,
             fancybox=True, shadow=True)
    
    # Adjust layout
    plt.tight_layout()
    
    # Save figure
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        print(f"\nMulti-cell plot saved to: {save_path}")
    
    plt.show()

In [ ]:
plot_multiple_cell_curves(
    all_adata=all_adata,
    cell_type_list=['Brain',
                    'Heart', 
                    'Cavity', 
                    'Spinal cord',
                    'Connective tissue',
                    'choroid plexus',
                    'Cartilage primordium',
                    'Meninges'],
    save_path='./multi_cell_growth.pdf',
    y_limits=(0.0, 0.3)
)

## Cell Number

In [ ]:
def plot_multiple_cell_numbers(all_adata, cell_type_list, save_path=None, 
                              y_limits=(0, 5000), line_width=2.5):
    """
    Plot cell numbers for multiple cell types on the same plot
    
    Parameters:
    -----------
    all_adata : dict
        Dictionary of AnnData objects by time
    cell_type_list : list
        List of cell type names to plot
    save_path : str, optional
        Path to save the figure
    y_limits : tuple, optional
        Y-axis limits (min, max) for cell numbers
    line_width : float, optional
        Width of the curves
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from scipy.interpolate import make_interp_spline
    import pandas as pd
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    
    # Define color palette
    colors = [
        '#2E86AB',  # Blue
        '#A23B72',  # Purple
        '#F18F01',  # Orange
        '#73AB84',  # Green
        '#C14953',  # Red
        '#6D435A',  # Dark Purple
        '#00A6A6',  # Teal
        '#F0B67F',  # Light Orange
        '#7D82B8',  # Lavender
        '#D56F3E',  # Brown
    ]
    
    # Store all curves for legend
    curve_handles = []
    curve_labels = []
    
    # Store all time points for consistent x-axis
    all_times = sorted(all_adata.keys())
    
    # Plot each cell type
    for i, cell_type in enumerate(cell_type_list):
        # Collect cell counts at each time point
        time_points = []
        cell_counts = []
        
        for time_point in all_times:
            adata = all_adata[time_point]
            
            # Count cells of this type at this time point
            if 'annotation' in adata.obs.columns:
                # Find cells matching this type
                cell_mask = adata.obs['annotation'].astype(str).str.contains(
                    cell_type, case=False, na=False
                )
                count = np.sum(cell_mask)
                
                if count > 0:  # Only add if we found cells
                    time_points.append(time_point)
                    cell_counts.append(count)
        
        if len(time_points) < 2:
            print(f"⚠ Not enough data points for: {cell_type}")
            continue
        
        # Convert to numpy arrays
        x = np.array(time_points)
        y = np.array(cell_counts)
        
        # Sort
        sort_idx = np.argsort(x)
        x_sorted = x[sort_idx]
        y_sorted = y[sort_idx]
        
        # Create smooth curve
        if len(x_sorted) > 3:
            # Smooth x values
            x_smooth = np.linspace(x_sorted.min(), x_sorted.max(), 500)
            
            # Cubic spline
            try:
                spline = make_interp_spline(x_sorted, y_sorted, k=3)
                y_smooth = spline(x_smooth)
                
                # Ensure no negative cell counts
                y_smooth = np.maximum(y_smooth, 0)
            except:
                # Polynomial fallback
                degree = min(3, len(x_sorted) - 1)
                coeffs = np.polyfit(x_sorted, y_sorted, degree)
                poly = np.poly1d(coeffs)
                y_smooth = poly(x_smooth)
                y_smooth = np.maximum(y_smooth, 0)
            
            # Plot the curve
            color = colors[i % len(colors)]
            line, = ax.plot(x_smooth, y_smooth, '-', 
                           color=color,
                           linewidth=line_width,
                           alpha=0.9,
                           label=f'{cell_type} (max: {int(y_smooth.max())})')
            
            curve_handles.append(line)
            curve_labels.append(cell_type)
            
            print(f"✓ Plotted: {cell_type} ({len(time_points)} time points, max: {int(y_smooth.max())} cells)")
        else:
            # Not enough points for smoothing, just connect them
            color = colors[i % len(colors)]
            line, = ax.plot(x_sorted, y_sorted, '-', 
                           color=color,
                           linewidth=line_width,
                           alpha=0.9,
                           label=f'{cell_type} (max: {int(y_sorted.max())})')
            
            curve_handles.append(line)
            curve_labels.append(cell_type)
            
            print(f"✓ Plotted: {cell_type} (points connected, max: {int(y_sorted.max())} cells)")
    
    # Set y-axis limits
    ax.set_ylim(y_limits)
    
    # Add horizontal line at y=0
    ax.axhline(y=0, color='black', linestyle='-', 
              linewidth=0.8, alpha=0.3, zorder=1)
    
    # Add subtle grid
    ax.grid(True, axis='y', alpha=0.15, linestyle='--', linewidth=0.5)
    ax.grid(True, axis='x', alpha=0.1, linestyle=':', linewidth=0.5)
    
    # Labels
    ax.set_xlabel('Time', fontsize=12, fontweight='bold')
    ax.set_ylabel('Cell Number', fontsize=12, fontweight='bold')
    
    # Title
    ax.set_title('Multiple Cell Types: Cell Number Over Time', 
                fontsize=14, fontweight='bold', pad=20)
    
    # Legend
    ax.legend(handles=curve_handles, labels=curve_labels,
             loc='best', fontsize=10, framealpha=0.9,
             fancybox=True, shadow=True)
    
    # Adjust layout
    plt.tight_layout()
    
    # Save figure
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        print(f"\nMulti-cell number plot saved to: {save_path}")
    
    plt.show()

In [ ]:
plot_multiple_cell_numbers(all_adata, cell_type_list=['Brain',
                    'Heart', 
                    'Cavity', 
                    'Spinal cord',
                    'Connective tissue',
                    'choroid plexus',
                    'Cartilage primordium',
                    'Meninges'
                    ], save_path='cell_number_t1_t2.pdf', 
                              y_limits=(0, 25000), line_width=2.5)

## Gene Expression

In [ ]:
import pandas as pd
pca_df = pd.read_csv('/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/mosta_pca_components_with_gene_names.csv')
pca_components = pca_df.iloc[:,1:].values
pca_df

In [ ]:
var_df = pd.DataFrame(index=pca_df['gene_short_name'].values)

In [ ]:
print(f"pca_components shape: {pca_components.shape}")

In [ ]:
import anndata
all_adata_new = []
for time_point, adata in all_adata.items():    
    X_embedding = adata.X
    X_embedding = X_embedding[:,2:]
    # 添加调试信息
    print(f"X_embedding shape: {X_embedding.shape}")

    X_ori = X_embedding @ pca_components.T
    
    adata_new = anndata.AnnData(
        X=X_ori,
        obs=adata.obs,
        var=var_df,
        uns=adata.uns,
        obsm=adata.obsm,
    )
    all_adata_new.append(adata_new)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter1d

def plot_celltype_gene_expression(all_adata, cell_type, 
                                  data_type='original',  # 新增：'original'或'zscore'
                                  sigma=1.0, 
                                  figsize=(16, 10), 
                                  show_top_genes=20,
                                  ylim=None):
    """
    绘制指定细胞类型的所有基因表达随时间变化的平滑曲线
    
    参数:
    ----------
    all_adata : list
        AnnData对象列表，每个元素代表一个时间点
    cell_type : str
        要可视化的细胞类型名称（如 'Microglia'）
    data_type : str, optional
        数据类型: 'original' (原始表达) 或 'zscore' (标准化表达)，默认'original'
    sigma : float, optional
        高斯平滑的标准差，默认1.0
    figsize : tuple, optional
        图形大小，默认(16, 10)
    show_top_genes : int, optional
        在图例中显示的基因数量，默认20个
    ylim : tuple or None, optional
        纵坐标范围，如(-3, 3)，None表示自动
    """
    
    print(f"=== 提取 {cell_type} 细胞数据 ===")
    print(f"数据类型: {data_type}")
    
    # 存储每个时间点的数据
    cell_expression_by_time = []  # 每个元素是 (time_point, gene_expression_vector)
    time_points = []
    cell_counts = []
    
    for i, adata in enumerate(all_adata):
        # 检查是否有注释信息
        if 'annotation' not in adata.obs.columns:
            print(f"警告: 时间点 {i} 没有 'annotation' 列")
            continue
        
        # 提取指定细胞类型的细胞
        cell_mask = adata.obs['annotation'] == cell_type
        cell_count = cell_mask.sum()
        
        if cell_count == 0:
            print(f"时间点 {i}: 没有{cell_type}细胞，跳过")
            continue
        
        # 获取时间点
        if 'time' in adata.obs.columns:
            current_time = adata.obs['time'].iloc[0]  # 取第一个细胞的时间
        else:
            current_time = i
        
        # 提取细胞的表达数据
        cell_adata = adata[cell_mask]
        
        # 计算该时间点的平均表达（按基因）
        avg_expression = cell_adata.X.mean(axis=0)  # 形状: (n_genes,)
        
        cell_expression_by_time.append((current_time, avg_expression))
        time_points.append(current_time)
        cell_counts.append(cell_count)
        
        print(f"  时间点 {current_time:.4f}: {cell_count} 个{cell_type}细胞")
    
    if not cell_expression_by_time:
        print(f"错误: 在所有时间点中都没有找到{cell_type}细胞！")
        return None
    
    # 转换为数组
    time_array = np.array(time_points)
    n_time_points = len(time_array)
    
    # 获取基因列表
    first_adata = all_adata[0]
    gene_names = first_adata.var_names.tolist()
    n_genes = len(gene_names)
    
    # 创建基因表达矩阵 (genes × time_points)
    expression_matrix = np.zeros((n_genes, n_time_points))
    
    for i, (time_point, exp_vector) in enumerate(cell_expression_by_time):
        expression_matrix[:, i] = exp_vector
    
    # 应用z-score标准化（如果需要）
    if data_type == 'zscore':
        print("应用z-score标准化...")
        zscore_matrix = np.zeros_like(expression_matrix)
        for i in range(n_genes):
            gene_data = expression_matrix[i]
            if gene_data.std() > 0:
                zscore_matrix[i] = (gene_data - gene_data.mean()) / gene_data.std()
            else:
                zscore_matrix[i] = gene_data  # 标准差为0，保持不变
        expression_matrix = zscore_matrix
    
    # 打印统计信息
    print(f"\n=== 数据统计 ===")
    print(f"细胞类型: {cell_type}")
    print(f"时间点数: {n_time_points}")
    print(f"基因数: {n_genes}")
    print(f"时间范围: {time_array[0]:.4f} - {time_array[-1]:.4f}")
    print(f"总细胞数: {sum(cell_counts)}")
    
    # 计算整体表达范围
    overall_min = expression_matrix.min()
    overall_max = expression_matrix.max()
    overall_mean = expression_matrix.mean()
    overall_std = expression_matrix.std()
    print(f"表达范围: [{overall_min:.3f}, {overall_max:.3f}]")
    print(f"整体均值: {overall_mean:.3f}")
    print(f"整体标准差: {overall_std:.3f}")
    
    # 绘制所有基因在同一图中的表达曲线
    print(f"\n=== 绘制表达曲线 ===")
    
    plt.figure(figsize=figsize)
    
    # 创建颜色映射
    colors = plt.cm.viridis(np.linspace(0, 1, n_genes))
    
    # 对每个基因绘制平滑曲线
    for i, gene in enumerate(gene_names):
        gene_exp = expression_matrix[i, :]
        
        # 应用平滑
        smoothed = gaussian_filter1d(gene_exp, sigma=sigma)
        
        # 绘制曲线
        plt.plot(time_array, smoothed, 
                 color=colors[i], 
                 alpha=0.6,  # 半透明以便看到重叠
                 linewidth=0.8,
                 label=gene if i < show_top_genes else None)  # 只标注前show_top_genes个基因
    
    # 设置图形属性
    plt.xlabel('Time', fontsize=14)
    
    # 根据数据类型设置y轴标签
    if data_type == 'zscore':
        ylabel = 'Normalized Expression (Z-score)'
    else:
        ylabel = 'Expression Level (Original)'
    
    plt.ylabel(ylabel, fontsize=14)
    
    title = f'{cell_type}: All Gene Expression Trajectories ({n_genes} genes)'
    if data_type == 'zscore':
        title += ' [Z-score Normalized]'
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    
    # 设置纵坐标范围
    if ylim is not None:
        plt.ylim(ylim)
    elif data_type == 'zscore':
        # 对于z-score数据，建议使用(-3, 3)范围
        suggested_min = max(-3, overall_min - 0.5)
        suggested_max = min(3, overall_max + 0.5)
        plt.ylim(suggested_min, suggested_max)
    
    # 添加图例（避免拥挤）
    if n_genes > 0 and show_top_genes > 0:
        plt.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.05, 1), ncol=1)
    
    # 添加网格
    plt.grid(True, alpha=0.2, linestyle='--')
    
    # 添加零线（对于z-score特别重要）
    plt.axhline(y=0, color='black', linewidth=0.8, alpha=0.5, linestyle='-')
    
    # 如果是z-score数据，添加标准差参考线
    if data_type == 'zscore':
        plt.axhline(y=1, color='gray', linewidth=0.5, alpha=0.3, linestyle='--')
        plt.axhline(y=-1, color='gray', linewidth=0.5, alpha=0.3, linestyle='--')
        plt.axhline(y=2, color='lightgray', linewidth=0.3, alpha=0.2, linestyle=':')
        plt.axhline(y=-2, color='lightgray', linewidth=0.3, alpha=0.2, linestyle=':')
        
        # 添加标准差区域填充
        plt.fill_between([time_array[0], time_array[-1]], -1, 1, 
                         alpha=0.05, color='gray', label='±1 SD region')
    
    # 调整布局
    plt.tight_layout()
    
    print("绘图完成！")
    
    # 返回数据用于进一步分析
    return {
        'expression_matrix': expression_matrix,
        'time_points': time_array,
        'gene_names': gene_names,
        'cell_counts': cell_counts,
        'data_type': data_type,
        'statistics': {
            'min': overall_min,
            'max': overall_max,
            'mean': overall_mean,
            'std': overall_std
        }
    }


In [ ]:
# Z-score标准化表达
result_zscore = plot_celltype_gene_expression(all_adata_new, 'Brain', data_type='zscore')
result_zscore = plot_celltype_gene_expression(all_adata_new, 'Brain', data_type='mea')
#自定义纵坐标范围
# result_custom = plot_celltype_gene_expression(all_adata_new, 'Microglia', data_type='zscore', ylim=(-2.5, 2.5))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.ndimage import gaussian_filter1d

def plot_gene_expression_heatmap(all_adata, cell_type, 
                                 color_scale='zscore',
                                 figsize=(16, 20),
                                 cluster_genes=True,
                                 cluster_timepoints=False,
                                 smooth_sigma=0.5,
                                 save_path=None,
                                 top_n_variable=50): # <--- 1. 新增参数: 默认只画Top 50
    """
    绘制指定细胞类型的基因表达热图 (支持筛选随时间变化最大的基因)
    """
    
    print(f"=== 绘制 {cell_type} 基因表达热图 ===")
    
    # --- 数据提取部分 (保持不变) ---
    time_data = []
    expr_data = []
    time_labels = []
    
    for i, adata in enumerate(all_adata):
        if 'annotation' not in adata.obs.columns:
            continue
        
        mask = adata.obs['annotation'] == cell_type
        n_cells = mask.sum()
        
        if n_cells == 0:
            continue
        
        # 获取时间
        if 'time' in adata.obs.columns:
            t = adata.obs['time'].iloc[0]
            time_label = f"T={t:.3f}"
        else:
            t = i
            time_label = f"T{i}"
        
        # 提取表达数据
        exp = adata[mask].X.mean(axis=0)
        
        time_data.append(t)
        expr_data.append(exp)
        time_labels.append(time_label)
    
    if not expr_data:
        print(f"错误: 未找到 {cell_type} 细胞！")
        return None
    
    # --- 准备矩阵 (保持不变) ---
    time_array = np.array(time_data)
    expr_matrix = np.vstack(expr_data).T  # 基因 × 时间点
    gene_names = np.array(all_adata[0].var_names.tolist()) # 转为numpy array方便索引
    n_genes, n_timepoints = expr_matrix.shape
    
    # --- 应用平滑 (保持不变) ---
    if smooth_sigma > 0:
        print(f"应用时间平滑 (sigma={smooth_sigma})...")
        for i in range(n_genes):
            expr_matrix[i] = gaussian_filter1d(expr_matrix[i], sigma=smooth_sigma)

    # =======================================================
    # ### --- 修改开始: 筛选随时间变化最大的 Top N 基因 --- ###
    # =======================================================
    if top_n_variable is not None and top_n_variable < n_genes:
        print(f"正在筛选随时间变化最大的 Top {top_n_variable} 基因...")
        
        # 1. 计算每个基因在时间维度上的方差 (axis=1 代表跨时间点计算)
        # 方差越大，说明该基因随时间的变化幅度越剧烈
        gene_variances = np.var(expr_matrix, axis=1)
        
        # 2. 获取方差最大的索引 (argsort默认从小到大，所以取最后top_n个)
        top_indices = np.argsort(gene_variances)[-top_n_variable:]
        
        # 3. 为了保持基因原本的顺序（可选），可以将索引再排个序
        # top_indices = np.sort(top_indices) 
        # 如果你希望热图里把变化最大的放一起，可以不sort，或者依赖后面的聚类功能
        
        # 4. 筛选矩阵和基因名
        expr_matrix = expr_matrix[top_indices, :]
        gene_names = gene_names[top_indices]
        n_genes = len(gene_names)
        
        print(f"筛选完成: 保留了 {n_genes} 个高变基因")
    # =======================================================
    # ### --- 修改结束 --- ###
    # =======================================================

    print(f"数据形状: {expr_matrix.shape} ({n_genes} 基因 × {n_timepoints} 时间点)")
    print(f"时间范围: {time_array.min():.3f} - {time_array.max():.3f}")

    # --- 颜色缩放 (保持不变) ---
    if color_scale == 'zscore':
        heatmap_data = np.zeros_like(expr_matrix)
        for i in range(n_genes):
            gene_data = expr_matrix[i]
            if gene_data.std() > 0:
                heatmap_data[i] = (gene_data - gene_data.mean()) / gene_data.std()
            else:
                heatmap_data[i] = gene_data
        color_label = 'Z-score'
        vmin, vmax = -2, 2
        
    elif color_scale == 'mean':
        heatmap_data = expr_matrix
        color_label = 'Mean Expression'
        vmin = np.percentile(heatmap_data, 5)
        vmax = np.percentile(heatmap_data, 95)
    
    # --- 聚类 (保持不变) ---
    if cluster_genes and n_genes > 1:
        print("对基因进行聚类...")
        gene_linkage = linkage(heatmap_data, method='average', metric='euclidean')
        gene_order = leaves_list(gene_linkage)
        heatmap_data = heatmap_data[gene_order, :]
        gene_labels = [gene_names[i] for i in gene_order]
    else:
        gene_labels = gene_names.tolist()
    
    if cluster_timepoints and n_timepoints > 1:
        # 通常画轨迹热图时不建议聚类时间点，因为时间是有序的
        print("对时间点进行聚类...")
        time_linkage = linkage(heatmap_data.T, method='average', metric='euclidean')
        time_order = leaves_list(time_linkage)
        heatmap_data = heatmap_data[:, time_order]
        time_labels = [time_labels[i] for i in time_order]
        time_array = time_array[time_order]
    
    # --- 绘图 (保持不变) ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, 
                                    gridspec_kw={'height_ratios': [15, 1]},
                                    constrained_layout=True)
    
    im = ax1.imshow(heatmap_data, 
                    aspect='auto',
                    cmap='RdBu_r', 
                    interpolation='nearest',
                    vmin=vmin,
                    vmax=vmax)
    
    ax1.set_yticks(np.arange(len(gene_labels)))
    ax1.set_yticklabels(gene_labels, fontsize=8) # 字体稍微调大一点，因为基因少了
    ax1.set_xticks(np.arange(len(time_labels)))
    ax1.set_xticklabels(time_labels, fontsize=9, rotation=45)
    
    ax1.set_xlabel('Time Points', fontsize=12)
    ax1.set_ylabel('Genes', fontsize=12)
    # 更新标题，显示Top N
    title_suffix = f"(Top {top_n_variable} Variable Genes)" if top_n_variable else "(All Genes)"
    ax1.set_title(f'{cell_type}: Gene Expression Heatmap {title_suffix}', 
                  fontsize=16, fontweight='bold', pad=20)
    
    # 颜色条
    cbar = plt.colorbar(im, cax=ax2, orientation='horizontal')
    cbar.set_label(color_label, fontsize=11)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"热图已保存到: {save_path}")
    
    plt.show()
    
    return {
        'heatmap_data': heatmap_data,
        'gene_labels': gene_labels,
        'time_labels': time_labels,
        'raw_matrix': expr_matrix # 返回筛选后的原始数据
    }

In [ ]:
zscoreresult = plot_gene_expression_heatmap(all_adata_new, 'Brain', color_scale='zscore', save_path='brain_top250_gene_heatmap_zscore.pdf', top_n_variable=250)
meanresult = plot_gene_expression_heatmap(all_adata_new, 'Brain', color_scale='mean', save_path='brain_top250_gene_heatmap_mean.pdf', top_n_variable=250)

In [ ]:
# 将所有数据整合到一个DataFrame中
data_dict = {
    'gene_labels': zscoreresult['gene_labels'],
    'heatmap_data': [zscoreresult['heatmap_data'][i] for i in range(len(zscoreresult['gene_labels']))],
    'raw_matrix': [zscoreresult['raw_matrix'][i] for i in range(len(zscoreresult['gene_labels']))]
}

df = pd.DataFrame(data_dict)
df.to_csv('brain_top250_gene_zscore_combined_data.csv', index=False)

In [ ]:
my_gene_set = [
    # --- Progenitors & Proliferation (增殖与祖细胞) ---
    'Fabp7',    # (BLBP) 放射状胶质细胞 (Radial Glia) 标记，大脑皮层发育的支架
    'Mki67',    # (Ki67) 细胞增殖标记，指示正在分裂的神经祖细胞

    # --- Neuronal Differentiation & Migration (神经元分化与迁移) ---
    'Sox11',    # 泛神经元转录因子，在神经前体细胞向神经元转化过程中高表达
    'Dcx',      # (Doublecortin) 迁移中的未成熟神经元，皮层板构建的关键
    'Tubb3',    # (TuJ1) 经典的早期神经元标记（beta-III tubulin）

    # --- Axon & Dendrite Growth (突触与网络构建) ---
    'Gap43',    # 轴突生长锥 (Growth Cones)，指示神经轴突的延伸
    'Stmn2',    # (SCG10) 神经元特异性微管调节蛋白，与轴突生长高度相关
    'Map2',     # 树突 (Dendrite) 标记，指示神经元逐渐成熟

    # --- Ventral Telencephalon / Interneurons (腹侧端脑与中间神经元) ---
    'Nr2f2',    # (COUP-TFII) 腹侧 CGE 来源的抑制性中间神经元标记
    'Meis2',    # 腹侧纹状体 (Striatum) 发育的关键转录因子，也是 GABA 能神经元分化标记
]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.cluster.hierarchy import linkage, leaves_list

def plot_custom_genes_fixed_order(all_adata, cell_type, gene_list,
                                  gene_order=None,  # 新增：强制指定基因顺序
                                  smooth_sigma=1.0,
                                  cluster_genes=False,  # 默认关闭聚类
                                  title_suffix=""):
    """
    分开显示：热图和趋势线分别在不同图形中
    可以强制指定基因顺序
    
    参数:
    ----------
    all_adata : list
        AnnData对象列表
    cell_type : str
        细胞类型名称
    gene_list : list
        要可视化的基因列表
    gene_order : list or None, optional
        强制指定的基因顺序，如果为None则使用输入顺序或聚类
    smooth_sigma : float, optional
        平滑参数
    cluster_genes : bool, optional
        是否对基因进行聚类（如果gene_order为None时生效）
    title_suffix : str, optional
        标题后缀
    """
    
    print(f"绘制 {cell_type} 自定义基因分析...")
    
    # 提取数据
    time_data, expr_data, valid_genes = [], [], []
    
    # 检查基因有效性
    for gene in gene_list:
        if gene in all_adata[0].var_names:
            valid_genes.append(gene)
        else:
            print(f"警告: 基因 {gene} 不存在，已跳过")
    
    if not valid_genes:
        print("错误: 没有有效的基因！")
        return None
    
    # 如果指定了gene_order，检查是否包含所有有效基因
    if gene_order is not None:
        # 检查gene_order中的基因是否都在valid_genes中
        missing_genes = [g for g in gene_order if g not in valid_genes]
        if missing_genes:
            print(f"警告: gene_order中的以下基因不存在: {missing_genes}")
        
        # 只保留有效的基因
        sorted_genes = [g for g in gene_order if g in valid_genes]
        
        # 添加缺失的基因到末尾
        extra_genes = [g for g in valid_genes if g not in sorted_genes]
        if extra_genes:
            print(f"添加缺失的基因到末尾: {extra_genes}")
            sorted_genes.extend(extra_genes)
            
        print(f"使用强制指定的基因顺序")
    elif cluster_genes:
        # 收集数据用于聚类
        temp_expr_data = []
        for adata in all_adata:
            if 'annotation' not in adata.obs.columns:
                continue
            
            mask = adata.obs['annotation'] == cell_type
            if mask.sum() == 0:
                continue
            
            exp_subset = adata[mask][:, valid_genes].X.mean(axis=0)
            temp_expr_data.append(exp_subset)
        
        if temp_expr_data:
            temp_matrix = np.vstack(temp_expr_data).T
            if smooth_sigma > 0:
                for i in range(len(valid_genes)):
                    temp_matrix[i] = gaussian_filter1d(temp_matrix[i], sigma=smooth_sigma)
            
            gene_linkage = linkage(temp_matrix, method='average', metric='euclidean')
            gene_order_indices = leaves_list(gene_linkage)
            sorted_genes = [valid_genes[i] for i in gene_order_indices]
            print(f"使用聚类后的基因顺序")
        else:
            sorted_genes = valid_genes
            print(f"无法聚类，使用输入顺序")
    else:
        sorted_genes = valid_genes  # 使用输入顺序
        print(f"使用输入顺序的基因顺序")
    
    print(f"最终基因顺序: {sorted_genes}")
    
    # 收集数据（按最终顺序）
    for adata in all_adata:
        if 'annotation' not in adata.obs.columns:
            continue
        
        mask = adata.obs['annotation'] == cell_type
        if mask.sum() == 0:
            continue
        
        t = adata.obs['time'].iloc[0] if 'time' in adata.obs.columns else len(time_data)
        
        # 按最终顺序提取基因表达
        exp_subset = adata[mask][:, sorted_genes].X.mean(axis=0)
        
        time_data.append(t)
        expr_data.append(exp_subset)
    
    if not expr_data:
        print(f"错误: 未找到 {cell_type} 细胞！")
        return None
    
    # 准备数据
    time_array = np.array(time_data)
    expr_matrix = np.vstack(expr_data).T  # 形状: (n_genes, n_timepoints)
    n_genes, n_timepoints = expr_matrix.shape
    
    print(f"分析 {n_genes} 个有效基因")
    print(f"时间点数量: {n_timepoints}")
    print(f"时间范围: {time_array[0]:.4f} - {time_array[-1]:.4f}")
    
    # ========== 图1: 双热图对比 ==========
    fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    # 左: Mean热图
    heatmap_mean = expr_matrix.copy()
    if smooth_sigma > 0:
        for i in range(n_genes):
            heatmap_mean[i] = gaussian_filter1d(heatmap_mean[i], sigma=smooth_sigma)
    
    vmin_mean = np.percentile(heatmap_mean, 5)
    vmax_mean = np.percentile(heatmap_mean, 95)
    
    # 绘制mean热图
    im1 = ax1.imshow(heatmap_mean, aspect='auto', cmap='OrRd',
                    vmin=vmin_mean, vmax=vmax_mean,
                    origin='upper')
    
    title1 = f'{cell_type}: Mean Expression'
    if title_suffix:
        title1 += f' - {title_suffix}'
    ax1.set_title(title1, fontsize=14, fontweight='bold')
    ax1.set_ylabel('Genes', fontsize=12)
    ax1.set_xlabel('Time', fontsize=12)
    
    # 设置y轴：按指定顺序的基因名称
    ax1.set_yticks(np.arange(n_genes))
    ax1.set_yticklabels(sorted_genes, fontsize=9)
    
    # 设置x轴：时间值
    if n_timepoints <= 15:
        ax1.set_xticks(np.arange(n_timepoints))
        ax1.set_xticklabels([f"{t:.2f}" for t in time_array], rotation=45, fontsize=9)
    else:
        step = max(1, n_timepoints // 8)
        indices = np.arange(0, n_timepoints, step)
        ax1.set_xticks(indices)
        ax1.set_xticklabels([f"{time_array[i]:.2f}" for i in indices], rotation=45, fontsize=9)
    
    # 右: Z-score热图
    heatmap_zscore = np.zeros_like(expr_matrix)
    for i in range(n_genes):
        gene_data = expr_matrix[i]
        if gene_data.std() > 0:
            heatmap_zscore[i] = (gene_data - gene_data.mean()) / gene_data.std()
    
    if smooth_sigma > 0:
        for i in range(n_genes):
            heatmap_zscore[i] = gaussian_filter1d(heatmap_zscore[i], sigma=smooth_sigma)
    
    im2 = ax2.imshow(heatmap_zscore, aspect='auto', cmap='RdBu_r',
                    vmin=-2, vmax=2,
                    origin='upper')
    
    title2 = f'{cell_type}: Z-score Normalized'
    if title_suffix:
        title2 += f' - {title_suffix}'
    ax2.set_title(title2, fontsize=14, fontweight='bold')
    ax2.set_yticks([])  # 隐藏y轴标签
    ax2.set_xlabel('Time', fontsize=12)
    
    # 设置x轴刻度（与左边保持一致）
    if n_timepoints <= 15:
        ax2.set_xticks(np.arange(n_timepoints))
        ax2.set_xticklabels([f"{t:.2f}" for t in time_array], rotation=45, fontsize=9)
    else:
        step = max(1, n_timepoints // 8)
        indices = np.arange(0, n_timepoints, step)
        ax2.set_xticks(indices)
        ax2.set_xticklabels([f"{time_array[i]:.2f}" for i in indices], rotation=45, fontsize=9)
    
    # 添加颜色条
    plt.colorbar(im1, ax=ax1, label='Mean Expression', shrink=0.8)
    plt.colorbar(im2, ax=ax2, label='Z-score', shrink=0.8)
    
    plt.suptitle(f'Gene Expression Heatmaps: {cell_type}{" - " + title_suffix if title_suffix else ""}', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    # ========== 图2: 原始表达平滑趋势线 ==========
    fig2, ax = plt.subplots(figsize=(16, 10))
    
    # 颜色映射
    if n_genes <= 20:
        colors = plt.cm.tab20(np.linspace(0, 1, n_genes))
    else:
        colors = plt.cm.rainbow(np.linspace(0, 1, n_genes))
    
    # 绘制每个基因的平滑曲线（按指定顺序）
    lines = []
    labels = []
    
    for i, gene_name in enumerate(sorted_genes):
        gene_data = expr_matrix[i]
        smoothed = gaussian_filter1d(gene_data, sigma=smooth_sigma)
        
        line, = ax.plot(time_array, smoothed,
                       color=colors[i % len(colors)],
                       linewidth=2.5,
                       alpha=0.8)
        lines.append(line)
        labels.append(f'{gene_name} (mean={gene_data.mean():.2f})')
    
    title3 = f'{cell_type}: Gene Expression Trends (Original Values)'
    if title_suffix:
        title3 += f' - {title_suffix}'
    ax.set_title(title3, fontsize=16, fontweight='bold')
    ax.set_xlabel('Time', fontsize=14)
    ax.set_ylabel('Expression Level (Original)', fontsize=14)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.axhline(y=0, color='black', linewidth=1, alpha=0.5)
    
    # 添加均值参考线
    overall_mean = expr_matrix.mean()
    ax.axhline(y=overall_mean, color='red', linewidth=1, alpha=0.5, 
               linestyle='--', label=f'Overall mean={overall_mean:.2f}')
    
    # 智能图例
    if n_genes > 10:
        ncol = 2 if n_genes > 20 else 1
        ax.legend(lines + [ax.get_lines()[-1]], labels + ['Overall mean'], 
                 fontsize=9, loc='upper left', 
                 bbox_to_anchor=(1.05, 1), ncol=ncol)
    else:
        ax.legend(lines + [ax.get_lines()[-1]], labels + ['Overall mean'], 
                 fontsize=10, loc='best')
    
    plt.tight_layout()
    plt.show()
    
    # ========== 图3: Z-score标准化后的平滑趋势线 ==========
    fig3, ax = plt.subplots(figsize=(16, 10))
    
    # 绘制每个基因的z-score平滑曲线
    zscore_lines = []
    zscore_labels = []
    
    for i, gene_name in enumerate(sorted_genes):
        zscore_data = heatmap_zscore[i]
        smoothed_zscore = gaussian_filter1d(zscore_data, sigma=smooth_sigma)
        
        line, = ax.plot(time_array, smoothed_zscore,
                       color=colors[i % len(colors)],
                       linewidth=2.5,
                       alpha=0.8)
        zscore_lines.append(line)
        
        # 计算z-score统计
        zscore_mean = zscore_data.mean()
        zscore_std = zscore_data.std()
        zscore_labels.append(f'{gene_name} (μ={zscore_mean:.2f}, σ={zscore_std:.2f})')
    
    title4 = f'{cell_type}: Z-score Normalized Expression Trends'
    if title_suffix:
        title4 += f' - {title_suffix}'
    ax.set_title(title4, fontsize=16, fontweight='bold')
    ax.set_xlabel('Time', fontsize=14)
    ax.set_ylabel('Expression Level (Z-score)', fontsize=14)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # 添加重要的参考线
    ax.axhline(y=0, color='black', linewidth=1.5, alpha=0.7, label='Mean (0)')
    ax.axhline(y=1, color='gray', linewidth=1, alpha=0.5, linestyle='--', label='+1 SD')
    ax.axhline(y=-1, color='gray', linewidth=1, alpha=0.5, linestyle='--', label='-1 SD')
    ax.axhline(y=2, color='lightgray', linewidth=0.8, alpha=0.3, linestyle=':')
    ax.axhline(y=-2, color='lightgray', linewidth=0.8, alpha=0.3, linestyle=':')
    
    # 添加标准差区域填充
    ax.fill_between(time_array, -1, 1, alpha=0.1, color='gray', label='±1 SD region')
    
    # 智能图例处理
    if n_genes <= 20:
        all_lines = zscore_lines + ax.get_lines()[-5:]
        all_labels = zscore_labels + ['Mean (0)', '+1 SD', '-1 SD', '±1 SD region']
        ax.legend(all_lines, all_labels, fontsize=9, loc='best')
    else:
        ax.legend(fontsize=10, loc='best')
        ax.text(0.02, 0.98, f'{n_genes} genes shown',
               transform=ax.transAxes,
               fontsize=10,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # ========== 返回数据 ==========
    return {
        'expression_matrix': expr_matrix,
        'heatmap_mean': heatmap_mean,
        'heatmap_zscore': heatmap_zscore,
        'time_points': time_array,
        'gene_names': valid_genes,
        'sorted_genes': sorted_genes,  # 返回实际使用的顺序
        'gene_order_used': sorted_genes,  # 明确说明使用的顺序
        'statistics': {
            'means': expr_matrix.mean(axis=1),
            'stds': expr_matrix.std(axis=1),
            'zscore_means': heatmap_zscore.mean(axis=1),
            'zscore_stds': heatmap_zscore.std(axis=1)
        }
    }

In [ ]:
result = plot_custom_genes_fixed_order(
    all_adata=all_adata_new,
    cell_type='Brain',
    gene_list=my_gene_set,  # 输入顺序
    gene_order=None,     # 不强制指定顺序
    smooth_sigma=1.0,
    cluster_genes=False,  # 关闭聚类，这样会保持输入顺序
    title_suffix="Input Order"
)

## Gene Expression Patterns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster, dendrogram
from scipy.ndimage import gaussian_filter1d
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
import re  # 需要导入正则模块
import warnings
warnings.filterwarnings('ignore')

# 如果要用gseapy进行通路分析
try:
    from gseapy import enrichr
    GSEAPY_AVAILABLE = True
except ImportError:
    print("未安装gseapy，将使用模拟数据进行演示")
    print("安装: pip install gseapy")
    GSEAPY_AVAILABLE = False

# === 配置本地数据库路径 ===
GSEA_DB_PATH = "/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/gsea_databases"

# 定义数据库名称到本地文件路径的映射
LOCAL_DATABASES = {
    'KEGG': f'{GSEA_DB_PATH}/KEGG_2019_Mouse.gmt',
    'GO': f'{GSEA_DB_PATH}/GO_Biological_Process_2021.gmt',
    'Reactome': f'{GSEA_DB_PATH}/Reactome_2016.gmt'
}

def clean_term_name(term, db_name):
    """
    清理Term名称，移除物种名和ID
    """
    # 转换为字符串以防万一
    term = str(term)
    
    # 删除"Homo sapiens"及之后的所有内容 (即使是Mouse数据，有时GMT文件可能保留了原始标记)
    if 'Homo sapiens' in term:
        term = term.split('Homo sapiens')[0].strip()
        # 如果以括号或冒号结尾，删除这些符号
        if term.endswith('(') or term.endswith(':') or term.endswith('-'):
            term = term[:-1].strip()
    
    # 删除"Mus musculus"及之后的所有内容 (针对小鼠数据)
    if 'Mus musculus' in term:
        term = term.split('Mus musculus')[0].strip()
        if term.endswith('(') or term.endswith(':') or term.endswith('-'):
            term = term[:-1].strip()
            
    # 如果是GO数据库，删除(GO:xxxx)部分
    if 'GO' in db_name:
        # 使用正则表达式删除(GO:xxxxx)格式的内容
        term = re.sub(r'\(GO:\d+\)', '', term).strip()
        
    return term

def analyze_gene_patterns_with_pathways(all_adata, cell_type, 
                                       n_clusters=8,
                                       method='average',
                                       distance_threshold=None,
                                       min_genes_per_cluster=5,
                                       organism='mouse',
                                       enrichment_db='KEGG', # 默认使用简写，会在内部映射到本地路径
                                       top_pathways_per_cluster=3,
                                       smooth_sigma=0.5):
    """
    提取基因表达模式并为每个模式进行通路富集分析
    """
    
    print(f"=== 分析 {cell_type} 细胞的基因表达模式与通路 ===")
    
    # 1. 提取所有时间点的表达数据
    expr_data, time_points, gene_names = extract_expression_data(
        all_adata, cell_type, smooth_sigma=smooth_sigma
    )
    
    if expr_data is None:
        print(f"错误: 未找到 {cell_type} 细胞的数据！")
        return None
    
    n_genes, n_timepoints = expr_data.shape
    print(f"数据形状: {n_genes} 基因 × {n_timepoints} 时间点")
    
    # 2. Z-score标准化（按基因）
    zscore_data = zscore_normalize_by_gene(expr_data)
    
    # 3. 层次聚类
    linkage_matrix, cluster_labels = perform_hierarchical_clustering(
        zscore_data, n_clusters=n_clusters, method=method
    )
    
    # 4. 提取每个模式的基因
    pattern_genes = extract_pattern_genes(gene_names, cluster_labels, min_genes_per_cluster)
    
    # 5. 计算每个模式的平均表达曲线
    pattern_curves = calculate_pattern_curves(zscore_data, cluster_labels)
    
    # 6. 通路富集分析 (使用修改后的函数)
    pathway_results = analyze_pattern_pathways_robust(
        pattern_genes, organism=organism, database=enrichment_db,
        top_n=top_pathways_per_cluster, min_genes=5
    )
    
    # 7. 整理所有结果
    results = {
        'cell_type': cell_type,
        'n_genes': n_genes,
        'n_timepoints': n_timepoints,
        'n_clusters': len(pattern_genes),
        'time_points': time_points,
        'gene_names': gene_names,
        'expr_matrix': expr_data,
        'zscore_matrix': zscore_data,
        'linkage_matrix': linkage_matrix,
        'cluster_labels': cluster_labels,
        'pattern_genes': pattern_genes,
        'pattern_curves': pattern_curves,
        'pathway_results': pathway_results,
        'parameters': {
            'n_clusters': n_clusters,
            'method': method,
            'organism': organism,
            'database': enrichment_db,
            'smooth_sigma': smooth_sigma
        }
    }
    
    # 8. 打印摘要信息
    print_summary(results)
    
    return results


def extract_expression_data(all_adata, cell_type, smooth_sigma=0.5):
    """
    从所有时间点提取指定细胞类型的表达数据
    """
    expr_data = []
    time_points = []
    
    for i, adata in enumerate(all_adata):
        if 'annotation' not in adata.obs.columns:
            continue
        
        mask = adata.obs['annotation'] == cell_type
        n_cells = mask.sum()
        
        if n_cells == 0:
            continue
        
        # 获取时间
        if 'time' in adata.obs.columns:
            t = float(adata.obs['time'].iloc[0])
        else:
            t = i
        
        # 提取表达数据
        exp = adata[mask].X.mean(axis=0)
        if hasattr(exp, 'A1'):  # 如果是稀疏矩阵
            exp = exp.A1
        
        time_points.append(t)
        expr_data.append(exp)
    
    if not expr_data:
        return None, None, None
    
    # 构建表达矩阵（基因 × 时间点）
    expr_matrix = np.vstack(expr_data).T
    gene_names = all_adata[0].var_names.tolist()
    
    # 时间平滑
    if smooth_sigma > 0:
        for i in range(len(gene_names)):
            expr_matrix[i] = gaussian_filter1d(expr_matrix[i], sigma=smooth_sigma)
    
    # 按时间排序
    time_order = np.argsort(time_points)
    time_points = np.array(time_points)[time_order]
    expr_matrix = expr_matrix[:, time_order]
    
    return expr_matrix, time_points, gene_names

def zscore_normalize_by_gene(expr_matrix):
    """
    对每个基因进行Z-score标准化
    """
    zscore_matrix = np.zeros_like(expr_matrix)
    n_genes = expr_matrix.shape[0]
    
    for i in range(n_genes):
        gene_data = expr_matrix[i]
        if gene_data.std() > 0:
            zscore_matrix[i] = (gene_data - gene_data.mean()) / gene_data.std()
        else:
            zscore_matrix[i] = gene_data
    
    return zscore_matrix

def perform_hierarchical_clustering(zscore_data, n_clusters=8, method='average'):
    """
    对基因进行层次聚类
    """
    print(f"使用 {method} 方法进行层次聚类...")
    
    # 计算距离矩阵
    from scipy.spatial.distance import pdist
    distance_matrix = pdist(zscore_data, metric='euclidean')
    
    # 层次聚类
    linkage_matrix = linkage(distance_matrix, method=method)
    
    # 提取聚类标签
    if n_clusters is not None:
        cluster_labels = fcluster(linkage_matrix, n_clusters, criterion='maxclust')
    else:
        # 使用距离阈值
        max_distance = linkage_matrix[-1, 2]
        threshold = max_distance * 0.3
        cluster_labels = fcluster(linkage_matrix, threshold, criterion='distance')
    
    return linkage_matrix, cluster_labels

def extract_pattern_genes(gene_names, cluster_labels, min_genes=5):
    """
    提取每个模式的基因列表
    """
    pattern_genes = {}
    unique_clusters = np.unique(cluster_labels)
    
    for cluster_id in unique_clusters:
        # 获取属于该聚类的基因索引
        gene_indices = np.where(cluster_labels == cluster_id)[0]
        
        if len(gene_indices) >= min_genes:
            # 提取基因名称
            genes = [gene_names[i] for i in gene_indices]
            pattern_genes[cluster_id] = {
                'gene_indices': gene_indices.tolist(),
                'genes': genes,
                'n_genes': len(genes)
            }
        else:
            print(f"警告: 聚类 {cluster_id} 只有 {len(gene_indices)} 个基因，跳过")
    
    print(f"提取到 {len(pattern_genes)} 个模式（每个至少 {min_genes} 个基因）")
    return pattern_genes

def calculate_pattern_curves(zscore_data, cluster_labels):
    """
    计算每个模式的平均表达曲线
    """
    pattern_curves = {}
    unique_clusters = np.unique(cluster_labels)
    
    for cluster_id in unique_clusters:
        # 获取属于该聚类的基因索引
        gene_indices = np.where(cluster_labels == cluster_id)[0]
        
        if len(gene_indices) > 0:
            # 计算平均曲线和标准差
            cluster_data = zscore_data[gene_indices, :]
            mean_curve = cluster_data.mean(axis=0)
            std_curve = cluster_data.std(axis=0)
            
            pattern_curves[cluster_id] = {
                'mean': mean_curve,
                'std': std_curve,
                'n_genes': len(gene_indices)
            }
    
    return pattern_curves

def analyze_pattern_pathways_robust(pattern_genes, organism='mouse', 
                                   database='KEGG', top_n=3, min_genes=5):
    """
    稳健的通路富集分析 (使用本地数据库并清理名称)
    """
    pathway_results = {}
    
    # 获取本地数据库路径，如果不在映射中，则假设传入的是路径或在线库名
    db_path = LOCAL_DATABASES.get(database, database)
    
    print(f"正在使用数据库: {db_path} (原始名称: {database})")
    
    for cluster_id, cluster_info in pattern_genes.items():
        genes = cluster_info['genes']
        n_genes = len(genes)
        
        print(f"\n分析模式 {cluster_id} ({n_genes} 个基因)...")
        
        if n_genes < min_genes:
            print(f"  基因数量太少 (<{min_genes})，跳过通路分析")
            pathway_results[cluster_id] = {
                'success': False,
                'message': f'Too few genes ({n_genes})',
                'significant_pathways': [],
                'top_pathways': []
            }
            continue
        
        try:
            # 使用gseapy进行通路富集分析
            if GSEAPY_AVAILABLE:
                # 注意：使用本地文件时，gseapy通常不需要organism参数，但为了兼容性可以保留
                enr = enrichr(gene_list=genes,
                            gene_sets=db_path, # 使用本地路径
                            organism=organism,
                            outdir=None,
                            cutoff=0.05)
                
                if enr.results is not None and not enr.results.empty:
                    # === 关键修改：在此处清理 Term 名称 ===
                    # 对整个 'Term' 列应用清理函数
                    enr.results['Term'] = enr.results['Term'].apply(
                        lambda x: clean_term_name(x, database)
                    )
                    
                    # 提取显著通路
                    sig_df = enr.results[enr.results['Adjusted P-value'] < 0.05].copy()
                    
                    if len(sig_df) > 0:
                        # 计算-log10(pvalue)
                        sig_df['-log10(pvalue)'] = -np.log10(sig_df['P-value'])
                        
                        # 获取top通路
                        top_pathways = []
                        for _, row in sig_df.head(top_n).iterrows():
                            # 这里的Term已经是清理过的了
                            term = row['Term']
                            pval = row['Adjusted P-value']
                            # 某些版本的gseapy Genes列可能是列表，也可能是分号分隔的字符串
                            genes_col = row['Genes']
                            if isinstance(genes_col, str):
                                genes_in_pathway = genes_col.split(';')
                            else:
                                genes_in_pathway = list(genes_col)
                            
                            top_pathways.append({
                                'term': term,
                                'p_value': float(pval),
                                '-log10(pvalue)': -np.log10(float(pval)),
                                'genes': genes_in_pathway,
                                'overlap_size': len(genes_in_pathway)
                            })
                        
                        pathway_results[cluster_id] = {
                            'success': True,
                            'n_significant': len(sig_df),
                            'significant_pathways': sig_df.to_dict('records'),
                            'top_pathways': top_pathways,
                            'all_results': enr.results
                        }
                        
                        print(f"  找到 {len(sig_df)} 个显著通路")
                        for i, pathway in enumerate(top_pathways[:3]):
                            print(f"    {i+1}. {pathway['term']} (p={pathway['p_value']:.2e})")
                    else:
                        pathway_results[cluster_id] = {
                            'success': False,
                            'message': 'No significant pathways found',
                            'significant_pathways': [],
                            'top_pathways': []
                        }
                        print("  无显著通路")
                else:
                    pathway_results[cluster_id] = {
                        'success': False,
                        'message': 'Enrichment analysis returned no results',
                        'significant_pathways': [],
                        'top_pathways': []
                    }
                    print("  通路分析无结果")
            else:
                # 模拟数据（用于演示）
                pathway_results[cluster_id] = {
                    'success': False,
                    'message': 'gseapy not installed',
                    'significant_pathways': [],
                    'top_pathways': []
                }
                print("  gseapy未安装，使用模拟数据")
                
        except Exception as e:
            print(f"  通路分析出错: {str(e)[:100]}...")
            pathway_results[cluster_id] = {
                'success': False,
                'message': f'Error: {str(e)[:100]}',
                'significant_pathways': [],
                'top_pathways': []
            }
    
    return pathway_results

def print_summary(results):
    """
    打印分析结果摘要
    """
    print("\n" + "="*60)
    print(f"分析结果摘要 - {results['cell_type']}")
    print("="*60)
    
    print(f"细胞类型: {results['cell_type']}")
    print(f"总基因数: {results['n_genes']}")
    print(f"时间点数: {results['n_timepoints']}")
    print(f"识别模式: {results['n_clusters']}")
    
    print(f"\n时间点: {results['time_points'].tolist()}")
    
    print(f"\n模式统计:")
    for cluster_id, cluster_info in results['pattern_genes'].items():
        n_genes = cluster_info['n_genes']
        
        # 检查通路分析结果
        pathway_success = False
        if cluster_id in results['pathway_results']:
            pathway_info = results['pathway_results'][cluster_id]
            pathway_success = pathway_info.get('success', False)
            n_sig_pathways = pathway_info.get('n_significant', 0)
        
        print(f"  模式 {cluster_id}: {n_genes} 个基因", end="")
        if pathway_success:
            print(f" - {n_sig_pathways} 个显著通路")
        else:
            print(" - 无显著通路")
    
    print("="*60)

def visualize_patterns_with_pathways(results, figsize=(20, 15), save_path=None):
    """
    可视化模式及其通路分析结果
    """
    cell_type = results['cell_type']
    pattern_genes = results['pattern_genes']
    pattern_curves = results['pattern_curves']
    pathway_results = results['pathway_results']
    time_points = results['time_points']
    
    n_clusters = len(pattern_genes)
    
    # 创建图形
    fig = plt.figure(figsize=figsize)
    
    # 根据聚类数量确定布局
    if n_clusters <= 4:
        n_rows = 2
        n_cols = 2
    elif n_clusters <= 6:
        n_rows = 2
        n_cols = 3
    else:
        n_rows = 3
        n_cols = 3
    
    gs = GridSpec(n_rows * 2, n_cols, hspace=0.4, wspace=0.3)
    
    # 为每个模式创建子图
    for i, (cluster_id, cluster_info) in enumerate(pattern_genes.items()):
        if i >= n_rows * n_cols:
            break
        
        row = (i // n_cols) * 2
        col = (i % n_cols)
        
        # 模式曲线图
        ax_curve = fig.add_subplot(gs[row:row+2, col])
        
        # 绘制平均曲线
        if cluster_id in pattern_curves:
            curve_info = pattern_curves[cluster_id]
            mean_curve = curve_info['mean']
            std_curve = curve_info['std']
            
            ax_curve.plot(time_points, mean_curve, 
                         linewidth=2, color=f'C{cluster_id-1}')
            ax_curve.fill_between(time_points, 
                                 mean_curve - std_curve,
                                 mean_curve + std_curve,
                                 alpha=0.3, color=f'C{cluster_id-1}')
        
        ax_curve.set_xlabel('Time', fontsize=10)
        ax_curve.set_ylabel('Z-score', fontsize=10)
        ax_curve.set_title(f'Pattern {cluster_id} ({cluster_info["n_genes"]} genes)', 
                          fontsize=12, fontweight='bold')
        ax_curve.grid(True, alpha=0.3)
    
    # 添加大标题
    plt.suptitle(f'{cell_type}: Gene Expression Patterns with Pathway Analysis',
                 fontsize=16, fontweight='bold', y=0.98)
    
    # 保存
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"结果图已保存到: {save_path}")
    
    plt.tight_layout()
    plt.show()

def save_analysis_results(results, output_dir='pattern_analysis_results'):
    """
    保存分析结果到文件
    """
    import os
    import json
    import pandas as pd
    import numpy as np

    # 定义一个辅助类处理 Numpy 类型的数据（防止保存 JSON 时数值报错）
    class NumpyEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, (np.int_, np.intc, np.intp, np.int8,
                                np.int16, np.int32, np.int64, np.uint8,
                                np.uint16, np.uint32, np.uint64)):
                return int(obj)
            elif isinstance(obj, (np.float_, np.float16, np.float32, np.float64)):
                return float(obj)
            elif isinstance(obj, (np.ndarray,)):
                return obj.tolist()
            return super(NumpyEncoder, self).default(obj)

    os.makedirs(output_dir, exist_ok=True)
    
    cell_type = results['cell_type']
    base_name = f"{cell_type.replace(' ', '_')}_patterns"

    # === 新增：计算所有基因的 Log2FoldChange (末点 / 起点) ===
    # results['expr_matrix'] 形状是 (n_genes, n_timepoints)
    # 第一列是第一个时间点，最后一列是最后一个时间点
    expr_matrix = results['expr_matrix']
    first_tp = expr_matrix[:, 0]
    last_tp = expr_matrix[:, -1]
    
    # 使用一个小值 epsilon 防止除以 0
    epsilon = 1e-6
    all_log2fc = np.log2((last_tp + epsilon) / (first_tp + epsilon))
    
    # 1. 保存模式基因列表
    pattern_genes = results['pattern_genes']
    for cluster_id, cluster_info in pattern_genes.items():
        # === 核心修复点：从 cluster_info 字典中获取之前存储的索引 ===
        current_indices = cluster_info['gene_indices'] 
        
        genes_df = pd.DataFrame({
            'gene': cluster_info['genes'],
            'pattern_id': cluster_id,
            'cell_type': cell_type,
            'logfoldchanges': all_log2fc[current_indices] # 使用提取出的索引
        })
        
        filename = f"{base_name}_cluster{cluster_id}_genes.csv"
        filepath = os.path.join(output_dir, filename)
        genes_df.to_csv(filepath, index=False)
    
    # 2. 保存模式曲线
    pattern_curves = results['pattern_curves']
    for cluster_id, curve_info in pattern_curves.items():
        curves_df = pd.DataFrame({
            'time': results['time_points'],
            'mean_zscore': curve_info['mean'],
            'std_zscore': curve_info['std']
        })
        
        filename = f"{base_name}_cluster{cluster_id}_curve.csv"
        filepath = os.path.join(output_dir, filename)
        curves_df.to_csv(filepath, index=False)
    
    # 3. 保存通路结果
    pathway_results = results['pathway_results']
    for cluster_id, pathway_info in pathway_results.items():
        if pathway_info.get('success', False) and 'all_results' in pathway_info:
            pathway_df = pathway_info['all_results']
            filename = f"{base_name}_cluster{cluster_id}_pathways.csv"
            filepath = os.path.join(output_dir, filename)
            pathway_df.to_csv(filepath, index=False)
    
    # 4. 保存摘要信息
    summary = []
    for cluster_id in pattern_genes.keys():
        summary_entry = {
            'pattern_id': cluster_id,
            'n_genes': pattern_genes[cluster_id]['n_genes'],
            'genes': ', '.join(pattern_genes[cluster_id]['genes'][:10]) + ('...' if len(pattern_genes[cluster_id]['genes']) > 10 else '')
        }
        
        if cluster_id in pathway_results and pathway_results[cluster_id].get('success', False):
            pathways = pathway_results[cluster_id].get('top_pathways', [])
            summary_entry['top_pathways'] = '; '.join([p['term'] for p in pathways[:3]])
            summary_entry['n_significant_pathways'] = pathway_results[cluster_id].get('n_significant', 0)
        else:
            summary_entry['top_pathways'] = 'None'
            summary_entry['n_significant_pathways'] = 0
        
        summary.append(summary_entry)
    
    summary_df = pd.DataFrame(summary)
    summary_path = os.path.join(output_dir, f"{base_name}_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    
    # 5. 保存完整结果（JSON格式）
    json_path = os.path.join(output_dir, f"{base_name}_full_results.json")
    
    # 简化结果以便保存为JSON
    simplified_results = {
        'cell_type': results['cell_type'],
        'n_genes': results['n_genes'],
        'n_timepoints': results['n_timepoints'],
        'n_clusters': results['n_clusters'],
        'time_points': results['time_points'].tolist() if hasattr(results['time_points'], 'tolist') else results['time_points'],
        'parameters': results['parameters'],
        'patterns': {}
    }
    
    for cluster_id in pattern_genes.keys():
        # 【关键修改】这里显式转换为 int 或 str，解决 keys must be str, int... 报错
        # 如果 cluster_id 是 numpy 类型，直接用作 key 会报错
        cid_key = int(cluster_id) 
        
        simplified_results['patterns'][cid_key] = {
            'n_genes': pattern_genes[cluster_id]['n_genes'],
            'gene_count': pattern_genes[cluster_id]['n_genes'],
            'genes_sample': pattern_genes[cluster_id]['genes'][:10]
        }
        
        if cluster_id in pathway_results and pathway_results[cluster_id].get('success', False):
            simplified_results['patterns'][cid_key]['pathways'] = pathway_results[cluster_id].get('top_pathways', [])[:3]
    
    with open(json_path, 'w') as f:
        # 使用自定义 Encoder 处理潜在的 numpy value 问题
        json.dump(simplified_results, f, indent=2, cls=NumpyEncoder)
    
    print(f"\n所有结果已保存到目录: {output_dir}/")
    print(f"  基因列表: {base_name}_cluster*_genes.csv")
    print(f"  表达曲线: {base_name}_cluster*_curve.csv")
    print(f"  通路结果: {base_name}_cluster*_pathways.csv")
    print(f"  分析摘要: {base_name}_summary.csv")
    print(f"  完整结果: {base_name}_full_results.json")

In [ ]:
# 执行分析
results = analyze_gene_patterns_with_pathways(
    all_adata=all_adata_new,
    cell_type='Brain',
    n_clusters=2,
    organism='mouse',
    enrichment_db='KEGG',
    top_pathways_per_cluster=3,
    smooth_sigma=0.5
)

# 可视化结果
if results:
    visualize_patterns_with_pathways(
        results,
        figsize=(18, 12),
        save_path='brain_patterns_pathways.pdf'
    )
    
    # 保存结果到文件
    save_analysis_results(results, output_dir='brain_pattern_analysis')
    
    # 查看特定模式的信息
    cluster_id = 1  # 查看第一个模式
    if cluster_id in results['pattern_genes']:
        print(f"\n=== 模式 {cluster_id} 详细信息 ===")
        print(f"基因数量: {results['pattern_genes'][cluster_id]['n_genes']}")
        print(f"前10个基因: {results['pattern_genes'][cluster_id]['genes'][:10]}")
        
        if cluster_id in results['pathway_results']:
            pathway_info = results['pathway_results'][cluster_id]
            if pathway_info.get('success', False):
                print(f"显著通路数量: {pathway_info.get('n_significant', 0)}")
                for pathway in pathway_info.get('top_pathways', [])[:3]:
                    print(f"  - {pathway['term']} (p={pathway['p_value']:.2e})")

In [ ]:
# ==============================================================================
# 1. 环境设置与 Nature 风格定义
# ==============================================================================
library(tidyverse)
library(clusterProfiler)
library(org.Mm.eg.db) # 小鼠注释库
library(enrichplot)
library(ggsci)        # 学术配色
library(cowplot)      # 拼图
library(showtext)     # 字体渲染

# 开启 showtext 以支持 Arial 字体 (Nature 要求)
showtext_auto()

# --- 定义 Nature Methods 风格主题 ---
theme_nature <- function(base_size = 10, base_family = "sans") {
  theme_bw(base_size = base_size, base_family = base_family) +
    theme(
      # 移除网格线 (或者保留极淡的网格)
      panel.grid.major = element_line(color = "grey92", size = 0.2),
      panel.grid.minor = element_blank(),
      
      # 坐标轴线条
      axis.line = element_line(color = "black", size = 0.5),
      panel.border = element_rect(color = "black", size = 0.8, fill = NA),
      
      # 字体设置 (Nature 通常要求 5-8 pt 的图例文本，8-10 pt 的轴标签)
      plot.title = element_text(size = 12, face = "bold", hjust = 0),
      axis.title = element_text(size = 10, face = "bold"),
      axis.text = element_text(size = 9, color = "black"),
      legend.title = element_text(size = 9, face = "bold"),
      legend.text = element_text(size = 8),
      
      # 图例位置优化
      legend.background = element_rect(fill = alpha("white", 0.5), color = NA),
      legend.key.size = unit(0.4, "cm")
    )
}

# ==============================================================================
# 2. 数据读取与预处理
# ==============================================================================

# 假设你的文件路径如下 (请根据实际情况修改)
work_dir <- "brain_pattern_analysis"
file_c1 <- file.path(work_dir, "Brain_patterns_cluster1_genes.csv")
file_c2 <- file.path(work_dir, "Brain_patterns_cluster2_genes.csv")

# 读取数据
c1_data <- read_csv(file_c1) %>% mutate(Cluster = "Pattern 1")
c2_data <- read_csv(file_c2) %>% mutate(Cluster = "Pattern 2")

# 合并数据
gene_df <- bind_rows(c1_data, c2_data)

# --- ID 转换 (Symbol -> Entrez ID) ---
# clusterProfiler 进行 KEGG 分析通常需要 Entrez ID，GO 分析两者皆可但推荐 Entrez
# 这是一个非常关键的步骤，因为有些基因名可能匹配不上
gene_ids <- bitr(gene_df$gene, 
                 fromType = "SYMBOL", 
                 toType = "ENTREZID", 
                 OrgDb = "org.Mm.eg.db")

# 将 Entrez ID 合并回原始数据框
merged_df <- gene_df %>%
  inner_join(gene_ids, by = c("gene" = "SYMBOL"))

print(paste("成功转换 ID 数:", nrow(merged_df)))
print(table(merged_df$Cluster))

# ==============================================================================
# 3. 使用 compareCluster 进行比较富集分析
# ==============================================================================
# 相比单独做两次 enrichGO，compareCluster 可以直接在同一张图上对比两个 Cluster
# 这对于展示模式差异非常有效

print("正在进行 GO 富集分析...")
# 3.1 GO 富集 (BP: Biological Process)
comp_GO <- compareCluster(ENTREZID ~ Cluster, 
                          data = merged_df, 
                          fun = "enrichGO",
                          OrgDb = "org.Mm.eg.db",
                          ont = "BP",  # 可以改为 "MF" 或 "CC"
                          pAdjustMethod = "BH",
                          pvalueCutoff = 0.05,
                          qvalueCutoff = 0.05,
                          readable = TRUE) # 结果自动转回 Gene Symbol

print("正在进行 KEGG 富集分析...")
# 3.2 KEGG 富集
comp_KEGG <- compareCluster(ENTREZID ~ Cluster, 
                            data = merged_df, 
                            fun = "enrichKEGG",
                            organism = "mmu", # 小鼠 code
                            pAdjustMethod = "BH",
                            pvalueCutoff = 0.05)

# ==============================================================================
# 4. Nature Methods 级别可视化
# ==============================================================================

# --- 辅助函数：处理过长的 Term 名称 ---
format_labels <- function(labels, width = 40) {
  str_wrap(labels, width = width)
}

# --- 图 1: 比较气泡图 (Comparative Dotplot) - 最经典 ---
# 这是一个展示两个 Cluster 功能差异的最直观方式
p_go <- dotplot(comp_GO, showCategory = 8) + 
  theme_nature() +
  scale_color_gsea() + # 使用 GSEA 风格配色，或者 scale_color_material("red")
  scale_y_discrete(labels = function(x) format_labels(x, 45)) +
  labs(title = "GO Biological Processes", 
       x = NULL, y = NULL, 
       size = "Gene Ratio", color = "p.adjust") +
  theme(axis.text.x = element_text(angle = 0, hjust = 0.5, face = "bold"))

p_kegg <- dotplot(comp_KEGG, showCategory = 8) + 
  theme_nature() +
  scale_color_material("blue") + 
  scale_y_discrete(labels = function(x) format_labels(x, 45)) +
  labs(title = "KEGG Pathways", 
       x = NULL, y = NULL, 
       size = "Gene Ratio", color = "p.adjust")

# --- 图 2: 基因-通路网络图 (Cnetplot) - 展示具体基因 ---
# Nature Methods 喜欢这种展示具体哪个基因导致了通路富集的图
# 注意：数据量大时，可以使用 circular = TRUE
p_net <- cnetplot(comp_GO, 
                  showCategory = 3, # 只展示前3个通路
                  foldChange = NULL, 
                  layout = "kk", 
                  color_category = "#E64B35FF", # 红色系
                  color_gene = "#4DBBD5FF",     # 蓝色系
                  node_label = "all") +
  theme_void() + # 网络图通常不需要坐标轴
  labs(title = "Gene-Concept Network (Pattern 1)") +
  theme(plot.title = element_text(size = 12, face = "bold"))

# 为了让 cnetplot 只展示 Pattern 1 (举例)，我们需要稍微 trick 一下
# 或者你可以分别对 comp_GO 的结果进行 filter 绘图

# ==============================================================================
# 5. 组合与保存
# ==============================================================================

# 使用 cowplot 拼接图形
# 左边放 GO 和 KEGG 的 Dotplot，右边放特定的网络图或其他分析
final_plot <- plot_grid(
  p_go, p_kegg, 
  labels = c("a", "b"), 
  label_size = 14,
  ncol = 2,
  align = 'h' # 顶部对齐
)

print(final_plot)

# --- 保存为 Publication Ready 格式 ---
# Nature 要求：
# 1. 矢量图 (PDF/EPS) 或高分辨率位图 (TIFF, >300dpi)
# 2. 宽度：单栏 89mm，双栏 183mm
# 3. 字体嵌入

output_file <- file.path(work_dir, "Brain_Gene_Enrichment_NatureMethods.pdf")

ggsave(filename = output_file, 
       plot = final_plot, 
       device = cairo_pdf, # 使用 cairo_pdf 确保字体嵌入更好
       width = 10, # 英寸，约 25cm，适合双栏大图
       height = 5, 
       dpi = 300)

print(paste("图表已保存至:", output_file))


### Pathway Annotation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.ndimage import gaussian_filter1d
from matplotlib.gridspec import GridSpec
import matplotlib.patches as patches
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import numpy as np
import seaborn as sns

def plot_nm_style_heatmap_pathway(results, 
                                  figsize=(16, 10), 
                                  top_n_pathways=3, 
                                  cmap='RdBu_r',
                                  save_name='NM_Style_Figure.pdf'):
    """
    绘制 Nature Methods 风格的组合图：左侧热图，右侧对齐的通路气泡。
    """
    
    # === 1. 准备绘图样式 (NM风格: Arial字体, 无衬线) ===
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = 7
    plt.rcParams['axes.linewidth'] = 0.8
    
    # === 2. 准备数据 ===
    # 提取并排序数据
    cluster_labels = results['cluster_labels']
    zscore_matrix = results['zscore_matrix']
    sorted_idx = np.argsort(cluster_labels)
    
    sorted_data = zscore_matrix[sorted_idx, :]
    sorted_labels = cluster_labels[sorted_idx]
    
    n_genes, n_time = sorted_data.shape
    unique_clusters = np.unique(sorted_labels)
    
    # 计算每个 cluster 的边界行号 (start, end)
    cluster_boundaries = {}
    current_start = 0
    for cid in unique_clusters:
        count = np.sum(sorted_labels == cid)
        cluster_boundaries[cid] = (current_start, current_start + count)
        current_start += count

    # === 3. 创建画布布局 ===
    fig = plt.figure(figsize=figsize, dpi=300)
    # 宽度比例: 热图 6.5 : 间隔 0.5 : 通路图 3 : Colorbar 0.2
    gs = gridspec.GridSpec(1, 4, width_ratios=[6.5, 0.1, 3.5, 0.2], wspace=0.05)
    
    ax_heatmap = fig.add_subplot(gs[0])
    ax_pathway = fig.add_subplot(gs[2])
    ax_cbar = fig.add_subplot(gs[3])
    
    # === 4. 绘制左侧热图 ===
    # 限制极值，保证颜色对比度
    vmin, vmax = -2.0, 2.0
    im = ax_heatmap.imshow(sorted_data, aspect='auto', cmap=cmap, 
                           vmin=vmin, vmax=vmax, interpolation='nearest')
    
    # X轴 (时间点)
    time_points = results['time_points']
    ax_heatmap.set_xticks(np.arange(n_time))
    ax_heatmap.set_xticklabels([f"{t:.1f}" for t in time_points], rotation=0, fontsize=8)
    ax_heatmap.set_xlabel("Time Points (days)", fontsize=9, fontweight='bold')
    
    # Y轴 (只显示 Cluster 分割线和标签)
    ax_heatmap.set_yticks([]) # 隐藏默认基因刻度
    ax_heatmap.set_ylabel(f"Genes (n={n_genes}) sorted by Pattern", fontsize=9, fontweight='bold')
    
    # 画分割线并标记 Pattern 编号
    for cid, (start, end) in cluster_boundaries.items():
        # 分割线
        ax_heatmap.axhline(end - 0.5, color='white', linewidth=1.5)
        
        # 在左侧边缘标记 Pattern ID
        center_y = (start + end) / 2
        ax_heatmap.text(-0.5, center_y, f"P{cid}", 
                        ha='right', va='center', 
                        transform=ax_heatmap.get_yaxis_transform(),
                        fontsize=10, fontweight='bold', color='#333333')

    # === 5. 绘制右侧通路气泡图 ===
    ax_pathway.set_ylim(n_genes, 0) # 翻转Y轴，与热图对齐
    ax_pathway.set_xlim(0, 1)
    ax_pathway.axis('off') # 隐藏边框
    
    pathway_results = results['pathway_results']
    
    # 气泡大小图例所需的最大最小值
    all_pvals = []
    
    # 遍历每个 Cluster 绘制气泡
    for cid in unique_clusters:
        start, end = cluster_boundaries[cid]
        cluster_height = end - start
        center_y = (start + end) / 2
        
        if cid not in pathway_results or not pathway_results[cid].get('success'):
            continue
            
        top_pw = pathway_results[cid].get('top_pathways', [])[:top_n_pathways]
        if not top_pw:
            continue
            
        # 动态计算行高 (防止 cluster 太小文字重叠)
        # 这里的 step 是相对于整个 Y 轴 (0~n_genes) 的像素高度
        # 稍微紧凑一点
        step = max(n_genes * 0.03, 10) 
        
        # 计算这组气泡的起始 Y
        # 使得这组气泡整体垂直居中于 Cluster 中心
        group_height = (len(top_pw) - 1) * step
        start_y = center_y - (group_height / 2)
        
        for i, pw in enumerate(top_pw):
            y_pos = start_y + i * step
            
            term = pw['term'].split('(')[0].strip() # 简化名字
            if len(term) > 30: term = term[:28] + "..."
            
            p_val = pw['p_value']
            all_pvals.append(p_val)
            
            # 计算气泡大小 (-log10 pvalue)
            nlogp = -np.log10(p_val)
            # 归一化大小，基础大小 + 增量
            size = 30 + (nlogp * 15)
            if size > 150: size = 150 # 上限
            
            # 颜色：可以使用单一高级灰，或者根据 P 值变色
            # 这里用一种 "Nature Blue"
            color = '#4575b4' 
            
            # 1. 绘制气泡点
            ax_pathway.scatter(0.1, y_pos, s=size, c=color, alpha=0.9, edgecolors='none')
            
            # 2. 绘制文字
            ax_pathway.text(0.18, y_pos, term, 
                            va='center', ha='left', fontsize=8, color='#2c3e50')
            
            # 3. (可选) 显示 P值
            # ax_pathway.text(0.95, y_pos, f"{p_val:.1e}", 
            #                 va='center', ha='right', fontsize=6, color='gray')

    # === 6. 添加 Colorbar ===
    cbar = plt.colorbar(im, cax=ax_cbar)
    cbar.set_label("Z-score Expression", fontsize=8)
    cbar.outline.set_visible(False) # 去掉Colorbar边框
    cbar.ax.tick_params(size=0) # 去掉刻度线
    
    # === 7. 添加气泡大小图例 (Legend) ===
    # 手动创建一个 legend 给气泡大小
    if all_pvals:
        min_p = min(all_pvals)
        max_p = max(all_pvals)
        
        # 创建虚拟点用于图例
        l1 = plt.scatter([],[], s=30 + (-np.log10(0.05)*15), c='#4575b4', label='p < 0.05')
        l2 = plt.scatter([],[], s=30 + (-np.log10(min_p)*15), c='#4575b4', label=f'p < {min_p:.1e}')
        
        ax_pathway.legend(handles=[l1, l2], title='Enrichment', 
                          bbox_to_anchor=(0.8, 1.05), # 放在顶部
                          loc='upper right', frameon=False, fontsize=7)

    plt.suptitle(f"{results['cell_type']} Temporal Patterns & Pathway Enrichment", 
                 y=0.95, fontsize=12, fontweight='bold')
    
    plt.savefig(save_name, bbox_inches='tight', transparent=True)
    print(f"Figure saved to {save_name}")
    plt.show()


In [ ]:
plot_nm_style_heatmap_pathway(
    results, 
    figsize=(12, 8),      # 调整为适合文章的尺寸
    top_n_pathways=5,     # 每个 Pattern 显示前4个通路
    save_name='Brain_pathway_heatmap.pdf' # 建议保存为 PDF 矢量图，方便放入 AI/CorelDraw 编辑
)

## LR Interaction

In [ ]:
# ==============================================================================
# 0. 基础设置与数据准备
# ==============================================================================
import pickle
import os
import numpy as np
import pandas as pd
import torch
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import FancyArrowPatch

# 设定保存路径
LR_RESULT_DIR = '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/evaluation/LR_analysis_1222'
os.makedirs(LR_RESULT_DIR, exist_ok=True)


In [ ]:
# 加载数据库
def load_lr_database():
    # 路径根据你的环境设定
    db_path = "/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/database/CellChatDB.ligrec.mouse.csv"
    if not os.path.exists(db_path):
        print(f"Warning: Database not found at {db_path}, using dummy.")
        return pd.DataFrame({'ligand': ['Mdk', 'App'], 'receptor': ['Ncl', 'Cd74'], 'pathway_name': ['MDK', 'APP']})
    return pd.read_csv(db_path)

In [ ]:
lr_db = load_lr_database()
db_genes = set(lr_db['ligand']).union(set(lr_db['receptor']))
len(db_genes)

In [ ]:
# ==============================================================================
# 1. 基因交集检查 (Intersection Check)
# ==============================================================================

# 使用 all_adata_new 中的第一个时间点来获取基因列表
# 注意：all_adata_new 是一个 list，包含了重建基因表达后的 adata
sample_adata = all_adata_new[2] 
gene_names = sample_adata.var_names.tolist()

adata_genes = set(gene_names)

intersect_genes = db_genes.intersection(adata_genes)
print(f"LR Database genes: {len(db_genes)}")
print(f"Generated Data genes: {len(adata_genes)}")
print(f"Intersection count: {len(intersect_genes)}")

# 筛选有效配受体对
valid_mask = lr_db.apply(lambda x: (x['ligand'] in adata_genes) and (x['receptor'] in adata_genes), axis=1)
lr_db_filtered = lr_db[valid_mask].copy()
print(f"Valid LR pairs for analysis: {len(lr_db_filtered)}")
lr_db_filtered


In [ ]:
# ==============================================================================
# 引入必要的绘图库并设置 Nature 风格
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import sys

# 设置全局绘图参数 (Nature Methods 风格)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans'] # 优先使用 Arial
plt.rcParams['pdf.fonttype'] = 42 # 保证导出 PDF 文字可编辑
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['axes.linewidth'] = 1.0 # 坐标轴线宽
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0
plt.rcParams['font.size'] = 10 # 基础字号


In [ ]:
import os
import gc
import pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.sparse import coo_matrix

In [ ]:
# 辅助函数：聚合 Attention 到 Cell Type
def analyze_attention_by_celltype(edge_index, attn, labels):
    edge_index = np.asarray(edge_index)
    attn = np.asarray(attn).astype(float)
    labels = np.asarray(labels)
    
    # 移除自环
    m = edge_index[0] != edge_index[1]
    send, recv, w = edge_index[0][m], edge_index[1][m], attn[m]

    types, type_id = np.unique(labels, return_inverse=True)
    T = len(types)
    n_per_type = np.bincount(type_id, minlength=T).astype(float)
    n_per_type[n_per_type == 0] = 1.0 

    # 聚合总强度
    M_sum = coo_matrix((w, (type_id[send], type_id[recv])), shape=(T, T)).toarray()
    
    # 每源细胞归一化 (Per-source Normalization)
    M_comm = M_sum / n_per_type[:, None]
     
    return M_comm, types

# 辅助函数：计算 LR Score (Type x Type)
def calculate_ct_lr_scores(comm_matrix, type_mean_expr, lr_db_filtered, gene_to_idx):
    lr_scores_dict = {}
    for idx, row in lr_db_filtered.iterrows():
        ligand, receptor = row['ligand'], row['receptor']
        lr_name = f"{ligand}_{receptor}"
        if ligand not in gene_to_idx or receptor not in gene_to_idx: continue
        
        l_idx, r_idx = gene_to_idx[ligand], gene_to_idx[receptor]
        L_means = type_mean_expr[:, l_idx]
        R_means = type_mean_expr[:, r_idx]
        
        # Formula: L_mean(Sender) * R_mean(Receiver) * Comm(Sender->Receiver)
        expr_product = L_means[:, None] * R_means[None, :]
        lr_score_matrix = expr_product * comm_matrix
        
        lr_scores_dict[lr_name] = lr_score_matrix
    return lr_scores_dict

In [ ]:
for i, adata_gene in enumerate(tqdm(all_adata_new, desc="Timepoints")):
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    time_key = adata_gene.obs['time'].iloc[0] # 获取时间标识
    
    # 1. 准备数据
    adata_embed = all_adata[time_key]
    data_val = adata_embed.X
    n_cells = data_val.shape[0]
    
    # 2. 计算 Attention (Model Inference)
    lnw0 = torch.log(torch.ones(n_cells, 1) / n_cells).to(device)
    data_tensor = torch.tensor(data_val, dtype=torch.float32).to(device)
    try: t_float = float(time_key)
    except: t_float = 0.0
    time_tensor = torch.tensor(t_float, dtype=torch.float32).to(device)

    f_net.eval()
    with torch.no_grad():
        _ = f_net.interaction_net(data_tensor, lnw0, time_tensor, return_attn=True)
        attn = f_net.interaction_net.gnn_layers[0].attn
        attn = torch.abs(attn).cpu().numpy().mean(axis=1) # 多头平均
        edge_index = f_net.interaction_net.edge_index.cpu().numpy()

    # 3. 计算 Communication Matrix
    if 'annotation' not in adata_gene.obs:
        print(f"Skipping {time_key}: No annotation found.")
        continue
    labels = adata_gene.obs['annotation'].values
    comm_matrix, cell_types = analyze_attention_by_celltype(edge_index, attn, labels)
    
    # 4. 基因表达严谨还原 (Rigorous Restoration)
    raw_expr = adata_gene.X
    if hasattr(raw_expr, "toarray"): raw_expr = raw_expr.toarray()
    
    # 空间变换：Log Space -> Count Space (expm1)
    # 假设输入数据已经做了 log(1+x)，这里还原回真实丰度
    # 这一步至关重要，因为 log(L)*log(R) 物理上无意义，L*R 才有意义
    expr_matrix_counts = np.expm1(raw_expr)
    
    # 5. 计算 Cell Type Mean Expression (在 Count 空间计算均值)
    # 注意：算术平均值在 Count 空间比 Log 空间更能反映真实的“群体总输出量”
    type_mean_expr = []
    for ct in cell_types:
        mask = (labels == ct)
        if mask.sum() > 0:
            type_mean_expr.append(expr_matrix_counts[mask, :].mean(axis=0))
        else:
            type_mean_expr.append(np.zeros(expr_matrix_counts.shape[1]))
    type_mean_expr = np.array(type_mean_expr)
    
    # 6. 计算并保存 LR Scores
    gene_to_idx = {g: idx for idx, g in enumerate(adata_gene.var_names)}
    lr_scores_dict = calculate_ct_lr_scores(comm_matrix, type_mean_expr, lr_db_filtered, gene_to_idx)
    
    # 保存结果
    with open(os.path.join(LR_RESULT_DIR, f'cell_types_{time_key}.pkl'), 'wb') as f:
        pickle.dump(cell_types, f)
    with open(os.path.join(LR_RESULT_DIR, f'lr_scores_{time_key}.pkl'), 'wb') as f:
        pickle.dump(lr_scores_dict, f)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pickle
import os

def plot_temporal_spatial_trajectory(
    lr_pair, 
    all_adata_new, 
    results_dir, 
    spatial_key='spatial',
    figsize_factor=3
):
    """
    绘制时空轨迹图 (Log-Scale Visualization).
    
    逻辑:
    1. L & R: 直接绘制 Log 表达量。
    2. Potential: log1p( Count(L) * Count(R) )。
    3. Score: log1p( Step2_Mass_Action_Score )。
    4. Colorbar: 使用 99.5% 分位数截断，确保对比度。
    """
    
    ligand, receptor = lr_pair.split('_')
    print(f"Plotting Log-Scale Trajectory for: {lr_pair} ...")

    # 容器：存储所有时间点的数据
    trajectory_data = {
        'time_keys': [],
        'coords': [],
        'ligand': [],   # Log Space
        'receptor': [], # Log Space
        'potential': [],# Log Space (Converted from Count product)
        'score': []     # Log Space (Converted from Step 2 result)
    }

    # --- 1. 数据收集循环 ---
    for adata in all_adata_new:
        time_key = adata.obs['time'].iloc[0]
        
        # A. 读取 Step 2 结果
        score_file = os.path.join(results_dir, f'lr_scores_{time_key}.pkl')
        type_file = os.path.join(results_dir, f'cell_types_{time_key}.pkl')
        
        # 容错：尝试浮点文件名
        if not os.path.exists(score_file):
            try:
                score_file = os.path.join(results_dir, f'lr_scores_{float(time_key):.2f}.pkl')
                type_file = os.path.join(results_dir, f'cell_types_{float(time_key):.2f}.pkl')
            except: pass
            
        if not os.path.exists(score_file):
            print(f"Skipping {time_key}: Results not found.")
            continue

        with open(score_file, 'rb') as f: lr_scores = pickle.load(f)
        with open(type_file, 'rb') as f: cell_types = pickle.load(f)

        if lr_pair not in lr_scores: continue

        # B. 提取坐标
        if spatial_key in adata.obsm:
            coords = adata.obsm[spatial_key]
        else:
            coords = np.zeros((adata.n_obs, 2))
            
        # C. 提取基因表达 (Log Space)
        # 假设 adata.X 已经是 log1p 后的数据
        l_expr_log = adata[:, ligand].X
        r_expr_log = adata[:, receptor].X
        if hasattr(l_expr_log, "toarray"): l_expr_log = l_expr_log.toarray().flatten()
        if hasattr(r_expr_log, "toarray"): r_expr_log = r_expr_log.toarray().flatten()
        
        # D. 计算 Potential (Log1p(Count * Count))
        # 1. 还原: Log -> Count (ReLU去除负值噪音)
        l_count = np.expm1(np.maximum(l_expr_log, 0))
        r_count = np.expm1(np.maximum(r_expr_log, 0))
        # 2. 物理乘积
        pot_count = l_count * r_count
        # 3. 可视化转换: Count -> Log1p
        pot_vis = np.log1p(pot_count)
        
        # E. 处理 Score (Log1p(Step2_Result))
        lr_matrix = lr_scores[lr_pair] # 这是物理 Count 级别的数值
        cell_scores = np.zeros(adata.n_obs)
        annotations = adata.obs['annotation'].values
        
        # 映射矩阵回细胞 (Vectorized)
        for i, ct in enumerate(cell_types):
            mask = (annotations == ct)
            if mask.sum() > 0:
                # 取 Receiver 端的总强度
                cell_scores[mask] = lr_matrix[:, i].sum() 
        
        # 可视化转换: Count -> Log1p
        # 解决长尾分布问题，让中等强度信号可见
        cell_scores_vis = np.log1p(cell_scores)

        # F. 存入列表
        trajectory_data['time_keys'].append(time_key)
        trajectory_data['coords'].append(coords)
        trajectory_data['ligand'].append(l_expr_log)
        trajectory_data['receptor'].append(r_expr_log)
        trajectory_data['potential'].append(pot_vis)
        trajectory_data['score'].append(cell_scores_vis)

    # --- 2. 内部绘图函数 ---
    def plot_row(data_list, metric_name, cmap, save_suffix):
        n_times = len(data_list)
        if n_times == 0: return

        # 计算全局统一的 Vmin / Vmax
        all_vals = np.concatenate(data_list)
        vmin = np.min(all_vals)
        
        # 关键: 使用 99.5% 分位数作为上限，忽略极端的 Outliers
        # 对于 Log 后的数据，这依然能保证对比度最佳
        vmax = np.percentile(all_vals, 99.5)
        
        if vmax == 0: vmax = 1.0
        if vmax <= vmin: vmax = vmin + 1e-5

        # 创建画布
        fig, axes = plt.subplots(1, n_times, figsize=(n_times * figsize_factor, figsize_factor))
        if n_times == 1: axes = [axes]
        fig.patch.set_facecolor("white")
        
        for i, ax in enumerate(axes):
            t_key = trajectory_data['time_keys'][i]
            coords = trajectory_data['coords'][i]
            vals = data_list[i]
            
            ax.set_facecolor("white")
            # 散点图
            sc = ax.scatter(coords[:, 0], coords[:, 1], c=vals, s=0.5, 
                            cmap=cmap, vmin=vmin, vmax=vmax, 
                            edgecolors='none', alpha=0.9, rasterized=True)
            
            ax.set_title(f"T: {t_key}", fontsize=10)
            ax.axis('off')
            ax.set_aspect('equal')
            
        # 添加 Colorbar
        plt.subplots_adjust(right=0.9)
        cbar_ax = fig.add_axes([0.92, 0.15, 0.01, 0.7]) 
        norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
        cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax)
        
        # 标注 Colorbar (注明 Log Scale)
        cb.set_label(f"{metric_name} (Log Scale)", fontsize=9)
        cb.outline.set_visible(False)
        
        # 保存
        save_name = f"{lr_pair}_Trajectory_{save_suffix}.pdf"
        full_path = os.path.join(results_dir, save_name)
        plt.savefig(full_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_name}")
        plt.close(fig)

    # --- 3. 执行绘制 4 组图 ---
    
    # 1. Ligand (Reds)
    plot_row(trajectory_data['ligand'], f"{ligand} Expression", "Reds", "Ligand")
    
    # 2. Receptor (Blues)
    plot_row(trajectory_data['receptor'], f"{receptor} Expression", "Blues", "Receptor")
    
    # 3. Potential (Plasma) - 代表理论上的共定位强度
    plot_row(trajectory_data['potential'], f"{lr_pair} Interaction Potential", "plasma", "Potential")
    
    # 4. Score (Viridis) - 代表模型预测的真实通信热点 (考虑了 CellType 和 Attention)
    plot_row(trajectory_data['score'], f"{lr_pair} Interaction Hotspot", "viridis", "Score")

# --- 调用示例 ---
# valid_pairs = list(lr_db_filtered['ligand'] + '_' + lr_db_filtered['receptor'])
# if len(valid_pairs) > 0:
#     plot_temporal_spatial_trajectory(valid_pairs[0], all_adata_new, LR_RESULT_DIR)

In [ ]:
# --- 调用示例 ---
# 假设你想画之前发现的那个 Pair
plot_temporal_spatial_trajectory(
    lr_pair='Ptprm_Ptprm',  
    all_adata_new=all_adata_new, 
    results_dir=LR_RESULT_DIR,
    spatial_key='spatial' # 确保你的 adata.obsm 里有这个 key
)

In [ ]:
plot_temporal_spatial_trajectory(
    lr_pair='Btc_Erbb4',  
    all_adata_new=all_adata_new, 
    results_dir=LR_RESULT_DIR,
    spatial_key='spatial' # 确保你的 adata.obsm 里有这个 key
)

In [ ]:
plot_temporal_spatial_trajectory(
    lr_pair='H2-Q10_Cd8b1',  
    all_adata_new=all_adata_new, 
    results_dir=LR_RESULT_DIR,
    spatial_key='spatial' # 确保你的 adata.obsm 里有这个 key
)

In [ ]:
plot_temporal_spatial_trajectory(
    lr_pair='Scgb3a2_Marco',  
    all_adata_new=all_adata_new, 
    results_dir=LR_RESULT_DIR,
    spatial_key='spatial' # 确保你的 adata.obsm 里有这个 key
)

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. 数据准备：读取 Step 2 的结果
# ==============================================================================
heatmap_data_list = []

# 按时间顺序遍历
for adata in all_adata_new:
    time_key = adata.obs['time'].iloc[0]
    
    # 构造文件名
    score_file = os.path.join(LR_RESULT_DIR, f'lr_scores_{time_key}.pkl')
    
    # 容错：检查文件
    if not os.path.exists(score_file):
        try: score_file = os.path.join(LR_RESULT_DIR, f'lr_scores_{float(time_key):.2f}.pkl')
        except: pass
    
    if not os.path.exists(score_file):
        print(f"Warning: Result file for time {time_key} not found. Skipping.")
        continue

    # 加载数据
    with open(score_file, 'rb') as f:
        lr_scores_dict = pickle.load(f)
    
    # 遍历字典中保存的所有 Pair (这些就是 lr_db_filtered 里的)
    for lr_name, score_matrix in lr_scores_dict.items():
        # 将矩阵求和，代表该时间点的全局强度
        total_score = np.sum(score_matrix)
        
        heatmap_data_list.append({
            'Time': time_key,
            'LR_Pair': lr_name,
            'Score': total_score
        })



In [ ]:
# ==============================================================================
# 2. 生成矩阵 & Z-Score 处理
# ==============================================================================

df_heatmap = pd.DataFrame(heatmap_data_list)

# Pivot: 行=LR Pair, 列=Time
# fillna(0) 很重要，防止某些时间点某个Pair完全没测到导致报错
lr_pivot = df_heatmap.pivot(index='LR_Pair', columns='Time', values='Score').fillna(0)

print(f"Plotting heatmap for {lr_pivot.shape[0]} pairs across {lr_pivot.shape[1]} timepoints.")

# Z-score 归一化 (Row-wise scaling)
# 目的：让强弱不同的 Pair 都能展示出趋势 (红=相对高, 蓝=相对低)
scaler = StandardScaler()
lr_zscore_vals = scaler.fit_transform(lr_pivot.T).T
lr_zscore = pd.DataFrame(lr_zscore_vals, index=lr_pivot.index, columns=lr_pivot.columns)

# ==============================================================================
# 3. 绘制 Clustermap (保持之前的配色风格)
# ==============================================================================
# row_cluster=True: 自动把趋势相似的 LR 聚在一起
# col_cluster=False: 严格保持时间轴顺序
g = sns.clustermap(
    lr_zscore,
    col_cluster=False, 
    row_cluster=True,
    cmap='RdBu_r',       # 经典的红蓝配色
    center=0,            # 0 (平均水平) 为白色
    figsize=(10, 10),     # 这里的长宽可以根据 Pair 的数量微调
    dendrogram_ratio=(0.15, 0.02),
    cbar_pos=(0.02, 0.85, 0.03, 0.10), # Colorbar 放左上角
    standard_scale=None  # 已手动 Z-score
)

# --- 细节美化 ---
# 轴标签
g.ax_heatmap.set_xlabel("Time", fontsize=12, fontweight='bold')
g.ax_heatmap.set_ylabel(f"Ligand Receptor Pairs", fontsize=12)

# 刻度标签
g.ax_heatmap.tick_params(axis='x', rotation=45, labelsize=10)

# 自动调整 Y 轴字体大小：如果 Pair 很多 (>50)，字号调小，否则字会重叠
y_fontsize = 10
if len(lr_zscore) > 50: y_fontsize = 6
if len(lr_zscore) > 100: y_fontsize = 4
g.ax_heatmap.tick_params(axis='y', labelsize=y_fontsize)

# Colorbar 标题
g.cax.set_title("Z-score", fontsize=10, pad=10)

# 保存
save_path = os.path.join(LR_RESULT_DIR, "LR_Dynamics_Clustermap_All.pdf")
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Heatmap saved to {save_path}")

plt.show()

    